# TII UAV — Audio-Only Characterization + standalone real-Mamba classifier

One **audio-only** notebook, run as a single Run All on Kaggle (GPU, internet on):

1. **Characterize** the TII `AudioBuffer` recordings correctly: 6 interleaved slots (5 channels + 1 empty slot),
   no sample-rate field, no per-chunk timestamps, 4 missions with corrupt `recording_started_time`.
2. **Visualize** the de-interleaved channels: structure, loudness over the flight, spectrograms, class and day effects.
3. **Train one standalone real-Mamba audio classifier** and report how strong audio is on its own, on a standard
   mission-level split and on **unseen recording days** (day-out).

**Out of scope:** IMU signal values, video, fusion, and comparison models. `SensorCombined` files are opened only to
read their first and last `timestamp` (the metadata audit in 1a).

**Descriptive vs. modelling.** Sections 2–3 are descriptive. Nothing found there is used to choose the model's inputs.
The representation in section 4 is fixed in advance: bins 1–16 of all 5 channels, and no data-dependent selection.

**Frequency axes** are per-channel FFT bins. The per-channel sampling rate is **unknown**: ~16 kHz per channel
(96 kHz interleaved, 16 ms per buffer) is an **unverified** hypothesis. It appears only as a secondary axis label and is
never used in a computation.

**Verification hooks 1–7** print PASS/FAIL. A FAIL in hooks 1–4 stops the notebook before any training.

## 0. Setup
- **0a:** environment (Kaggle / Colab / local), output paths, and the dataset. An attached Kaggle dataset is used if
  one is found; otherwise the repo is cloned with `git clone --depth 1`.
- **0b:** real `mamba-ssm`. There is no fallback block: the notebook stops with a clear error if it cannot be imported.
- **0c:** every constant, in one cell.
- **0d:** seeding and determinism flags, the dataset-file access logger, the PASS/FAIL hook helper and the figure helpers.

In [ ]:
# ==============================================================================
# 0a. ENVIRONMENT, PATHS, DATASET
# ==============================================================================
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")   # must be set before the first cuBLAS call (determinism)

import ast
import copy
import csv
import gc
import glob
import hashlib
import json
import math
import random
import re
import subprocess
import sys
import time
from collections import Counter, defaultdict
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

IS_KAGGLE = Path("/kaggle/working").exists()
IS_COLAB = (not IS_KAGGLE) and ("google.colab" in sys.modules or Path("/content").exists())
if IS_KAGGLE:
    BASE, OUT = Path("/kaggle/working/tii"), Path("/kaggle/working")
elif IS_COLAB:
    BASE, OUT = Path("/content/tii"), Path("/content")
else:
    BASE, OUT = Path.cwd() / "tii", Path.cwd() / "audio_only_outputs"
BASE.mkdir(parents=True, exist_ok=True)
OUT.mkdir(parents=True, exist_ok=True)
ENV = "Kaggle" if IS_KAGGLE else ("Colab" if IS_COLAB else "local")

REPO_URL = "https://github.com/tiiuae/UAV-Realistic-Fault-Dataset.git"


def _is_dataset_root(p: Path) -> bool:
    return p.is_dir() and all((p / str(c)).is_dir() for c in range(5))


_attached = []
for _pat in ("/kaggle/input/*/Dataset", "/kaggle/input/*/*/Dataset", "/kaggle/input/*/*/*/Dataset"):
    _attached += [Path(p) for p in sorted(glob.glob(_pat)) if _is_dataset_root(Path(p))]
if _attached:
    DATA_ROOT, DATA_SOURCE = _attached[0], "attached Kaggle dataset"
else:
    _repo = BASE / "UAV-Realistic-Fault-Dataset"
    if not _is_dataset_root(_repo / "Dataset"):
        print(f"Cloning {REPO_URL} (--depth 1) into {_repo} ...")
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(_repo)], check=True)
    DATA_ROOT, DATA_SOURCE = _repo / "Dataset", "git clone --depth 1"
assert _is_dataset_root(DATA_ROOT), f"{DATA_ROOT} does not contain class folders 0..4"

FIG_DIR = OUT / "figures"
CACHE_DIR = OUT / "cache"
PROBS_DIR = OUT / "audio_only_probs"
CKPT_DIR = OUT / "audio_only_checkpoints"
RESULTS_CSV = OUT / "audio_only_results.csv"
for _d in (FIG_DIR, CACHE_DIR, PROBS_DIR, CKPT_DIR):
    _d.mkdir(parents=True, exist_ok=True)

print(f"Environment: {ENV} | BASE {BASE} | outputs {OUT}")
print(f"Dataset: {DATA_ROOT} ({DATA_SOURCE})")
print(f"Figures -> {FIG_DIR} | cache -> {CACHE_DIR} | results -> {RESULTS_CSV} | probs -> {PROBS_DIR}")

In [ ]:
# ==============================================================================
# 0b. REAL mamba-ssm (no fallback). Installs only if it is not importable already.
# ==============================================================================
import importlib
import importlib.util

import torch


def _pip(args):
    print("  pip install " + " ".join(args))
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], capture_output=True, text=True)
    if r.returncode != 0:
        print((r.stderr or r.stdout)[-2000:])
    return r.returncode == 0


if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU detected. Real mamba-ssm needs a GPU: enable a GPU accelerator (Kaggle: Settings -> "
                       "Accelerator -> GPU) and re-run. There is deliberately no CPU fallback in this notebook.")

if importlib.util.find_spec("mamba_ssm") is None:
    print(f"Installing mamba-ssm (torch {torch.__version__}, CUDA {torch.version.cuda}, python "
          f"{sys.version_info.major}.{sys.version_info.minor}) ...")
    _ok = (_pip(["packaging", "ninja"])
           and _pip(["causal-conv1d>=1.2.0", "--no-build-isolation"])
           and _pip(["mamba-ssm", "--no-build-isolation"]))
    importlib.invalidate_caches()
    if not _ok:
        raise RuntimeError("mamba-ssm installation FAILED (see pip output above). Check that internet is ON and a GPU "
                           "is attached, then restart the session and re-run. No fallback model is used.")

try:
    import mamba_ssm
    from mamba_ssm import Mamba as MAMBA_CLASS
except Exception as exc:
    raise RuntimeError(f"mamba_ssm could not be imported ({exc!r}). Restart the kernel after installation and re-run; "
                       "no fallback model is used.") from exc
print(f"mamba_ssm {getattr(mamba_ssm, '__version__', '?')} imported: {MAMBA_CLASS.__module__}.{MAMBA_CLASS.__name__} | "
      f"torch {torch.__version__} | CUDA {torch.version.cuda} | GPU {torch.cuda.get_device_name(0)}")

In [ ]:
# ==============================================================================
# 0c. CONSTANTS — every fixed value lives here
# ==============================================================================
SEEDS = [42, 123, 2024]

# --- AudioBuffer layout: 1536 = 256 frames x 6 interleaved slots; sample i -> frame i // 6, slot i % 6 ---
N_SLOTS = 6
DROP_SLOT = 5                                   # 0-based slot index 5 (the 6th slot): identically zero, dropped
KEEP_SLOTS = [s for s in range(N_SLOTS) if s != DROP_SLOT]
N_CH = len(KEEP_SLOTS)                          # 5 audio channels
FRAME = 256                                     # frames per buffer = samples per channel per buffer
BUFFER_LEN = N_SLOTS * FRAME                    # 1536
N_RFFT = FRAME // 2 + 1                         # 129 per-channel rFFT bins (0..128)
KEEP_BINS = list(range(1, 17))                  # model input: per-channel bins 1..16 (fixed a priori)
N_FEAT = N_CH * len(KEEP_BINS)                  # 80

# --- windows (chunk counts only: no IMU, no time axis) ---
WINDOW_CHUNKS = 128
STRIDE_CHUNKS = 64
TA = 64                                         # tokens per window
TOKEN_POOL = WINDOW_CHUNKS // TA                # 2 consecutive chunks averaged per token
STEADY = (0.10, 0.90)                           # steady flight = 10%..90% of a mission's chunks (by position)

ASSUMED_CHANNEL_RATE_HZ = 16000                 # UNVERIFIED, used ONLY for secondary axis labels, never in any computation

# --- dataset facts (asserted in section 1) ---
N_CLASSES = 5
EXCLUDE_RATIO = 0.5                             # exclude if audio_start_offset / mission_duration >= this
EXPECTED_EXCLUDED = {"rosbag2_2021_12_29-09_10_31", "rosbag2_2021_12_31-10_12_31",
                     "rosbag2_2021_12_31-09_58_13", "rosbag2_2021_12_31-10_04_22"}
EXPECTED_N_MISSIONS = 95
EXPECTED_CLASS_COUNTS = [18, 19, 18, 20, 20]
DAYS = ["2021-12-22", "2021-12-31", "2022-01-04", "2022-01-05", "2022-01-10"]
DAY_MERGE = {"2021-12-29": "2021-12-31"}        # the single remaining 12-29 mission joins the 12-31 group
EXPECTED_DAY_COUNTS = [10, 36, 17, 15, 17]
SMALL_FOLD_DAY = "2021-12-22"

# --- model + training: transcribed from version 28/minor-project-v28.ipynb (config cell, UnimodalMamba, train_unimodal),
#     the audio arms A1/A2. Re-checked against that file at runtime in section 6 if it is attached. ---
D_MODEL = 256
DROPOUT = 0.15
MAMBA_NUM_BLOCKS = 2
MAMBA_D_STATE = 16
MAMBA_D_CONV = 2
MAMBA_EXPAND = 2
EXPECTED_MAMBA_PARAMS = 874_496                 # 2 x 437,248 (real mamba_ssm, incl. the pre-norm LayerNorms)
EPOCHS = 20                                     # fixed; no early stopping in v28
BATCH_SIZE = 32
LEARNING_RATE = 1e-4                            # AdamW
WEIGHT_DECAY = 1e-4
LR_MIN = 1e-6                                   # CosineAnnealingLR(T_max=EPOCHS, eta_min=LR_MIN), stepped per epoch
GRAD_CLIP_NORM = 1.0
NUM_WORKERS = 2
# checkpoint rule (v28): after every epoch, keep the state with the highest validation macro-F1 (strictly greater)

# --- splits ---
SPLIT_SEED, TRAIN_FRACTION, VAL_FRACTION = 42, 0.70, 0.15          # v24 split_missions
INNER_VAL_FRACTION, INNER_VAL_SEED = 0.15, 2027                    # v27 _split_inner_val (day-out protocol index 0)
INNER_VAL_FALLBACK_SEED = 2026                                     # used ONLY if v27's notebook cannot be read
V24_SPLIT_SHA1 = "e60744ab660df986"             # sha1[:16] of v24's split_missions source (ast segment)
V27_INNER_VAL_SHA1 = "58e3c037c513ace2"         # sha1[:16] of v27's _split_inner_val source (ast segment)
V24_CORRECTED_SPLIT_COUNTS = {"train": 67, "val": 13, "test": 15}   # v24 run output after the 4 exclusions

# --- analysis (fixed before seeing results) ---
N_PERMUTATIONS = 1000
PERM_SEED = 2029
BOOTSTRAP_RESAMPLES = 2000
BOOTSTRAP_SEED = 2028
TOP_CELLS = 10
SIGN_CONSISTENT_MIN_DAYS = 4
MIN_PER_CLASS_FOR_DAY_SIGN = 2
SPEARMAN_THRESHOLD = 0.7
PCA_BINS = list(range(1, 33))
EVIDENCE_K = [1, 2, 4, 8, 16, 32]
V28_A1_DAYOUT_F1 = (0.741, 0.689, 0.791)        # v28 audio arm A1, day-out pooled macro-F1 [95% CI] — context only

CHECK_DETERMINISM = True                        # train seed 42 for 3 epochs twice and compare losses exactly
DETERMINISM_EPOCHS = 3

HZ_PER_BIN = ASSUMED_CHANNEL_RATE_HZ / FRAME    # 62.5 Hz/bin — secondary axis labels ONLY
print(f"{N_SLOTS} slots x {FRAME} frames = {BUFFER_LEN} samples per buffer | channels kept {[s + 1 for s in KEEP_SLOTS]} "
      f"(slot {DROP_SLOT + 1} dropped) | model input {N_CH} ch x bins {KEEP_BINS[0]}-{KEEP_BINS[-1]} = {N_FEAT} features")
print(f"Windows: {WINDOW_CHUNKS} chunks, stride {STRIDE_CHUNKS}, {TA} tokens of {TOKEN_POOL} chunks | seeds {SEEDS}")

In [ ]:
# ==============================================================================
# 0d. SEEDING / DETERMINISM, FILE-ACCESS LOGGER, PASS/FAIL HOOKS, FIGURE HELPERS
# ==============================================================================
def set_all_seeds(seed: int) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def make_generator(seed: int) -> torch.Generator:
    """DataLoader shuffling uses its own generator, so the shuffle order depends only on the seed."""
    g = torch.Generator()
    g.manual_seed(seed)
    return g


def worker_init_fn(worker_id: int) -> None:
    s = torch.initial_seed() % 2 ** 32
    np.random.seed(s)
    random.seed(s)


torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True, warn_only=True)
set_all_seeds(SEEDS[0])
print("Determinism settings: cudnn.deterministic=True, cudnn.benchmark=False, "
      "torch.use_deterministic_algorithms(True, warn_only=True) (warn_only because mamba-ssm's custom CUDA kernels are "
      f"outside PyTorch's deterministic registry), CUBLAS_WORKSPACE_CONFIG={os.environ['CUBLAS_WORKSPACE_CONFIG']}, "
      "per-run seeding of random/numpy/torch/CUDA, a seeded torch.Generator for DataLoader shuffling, and worker_init_fn.")

# ---- dataset-file access logger: EVERY dataset file read in this notebook goes through open_logged ----
ACCESS_KINDS = ("AudioBuffer:first_record", "AudioBuffer:full_parse", "SensorCombined:first_last_timestamp")
FILE_LOG = Counter()
FILE_LOG_PER_MISSION = {k: Counter() for k in ACCESS_KINDS}


def open_logged(path, kind, mode="r"):
    assert kind in ACCESS_KINDS, f"unknown access kind {kind}"
    FILE_LOG[kind] += 1
    FILE_LOG_PER_MISSION[kind][Path(path).parent.name] += 1
    return open(path, mode) if "b" in mode else open(path, mode, encoding="utf-8")


# ---- verification hooks ----
HOOKS = defaultdict(list)
HOOK_NAMES = {1: "mission/class/day counts + exclusion set", 2: "slot 6 identically zero", 3: "split leakage + train-only z-score",
              4: "windows: 128 chunks, within mission, tokens (64, 80)", 5: "file-access summary",
              6: "real mamba_ssm + params + determinism", 7: "figure frequency-axis caveats"}


def hook(n, ok, msg, stop=None):
    ok = bool(ok)
    HOOKS[n].append((ok, msg))
    print(f"[HOOK {n}] {'PASS' if ok else 'FAIL'} — {msg}")
    stop = (n <= 4) if stop is None else stop
    if not ok and stop:
        raise RuntimeError(f"HOOK {n} FAILED: {msg}. Stopping before anything is trained.")
    return ok


# ---- figures (palette: validated categorical order; sequential = one hue; diverging = two hues + neutral midpoint) ----
CLASS_COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4"]
CLASS_MARKERS = ["o", "s", "^", "D", "v"]
DAY_COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4"]
DAY_MARKERS = ["o", "s", "^", "D", "v"]
SEQ_CMAP, DIV_CMAP = "Blues", "RdBu_r"
plt.rcParams.update({"figure.dpi": 100, "axes.spines.right": False, "axes.grid": True, "grid.alpha": 0.25,
                     "grid.linewidth": 0.6, "lines.linewidth": 1.5, "axes.titlesize": 10, "axes.labelsize": 9,
                     "xtick.labelsize": 8, "ytick.labelsize": 8, "legend.fontsize": 8})
FREQ_LABEL = "per-channel FFT bin (1-128)"
HZ_LABEL = "approx. Hz if the per-channel rate were 16 kHz (unverified)"
FREQ_CAVEAT = ("Frequency axis = per-channel FFT bin of the de-interleaved channels. The Hz axis assumes a per-channel "
               "rate of 16 kHz, which is UNVERIFIED (the files carry no sample rate).")
FIGURES = []


def add_bin_axis(ax, axis="x"):
    """Secondary Hz axis for a per-channel-bin axis. Label only: HZ_PER_BIN is never used in a computation."""
    fwd, inv = (lambda b: b * HZ_PER_BIN), (lambda h: h / HZ_PER_BIN)
    if axis == "x":
        sec = ax.secondary_xaxis("top", functions=(fwd, inv))
        sec.set_xlabel(HZ_LABEL, fontsize=7)
    else:
        sec = ax.secondary_yaxis("right", functions=(fwd, inv))
        sec.set_ylabel(HZ_LABEL, fontsize=7)
    sec.tick_params(labelsize=7)
    ax._bin_caveat = True
    return sec


def save_fig(fig, name, caveat, freq_axes=()):
    """Every figure is saved as PNG with its caveat printed under it. Frequency figures must carry the Hz caveat."""
    assert caveat, f"{name}: every figure needs a caveat"
    freq_ok = all(getattr(ax, "_bin_caveat", False) for ax in freq_axes) and (not freq_axes or "UNVERIFIED" in caveat)
    fig.text(0.5, -0.01, caveat, ha="center", va="top", fontsize=8, color="#52514e", wrap=True)
    fig.savefig(FIG_DIR / f"{name}.png", dpi=130, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    FIGURES.append({"name": name, "freq": bool(freq_axes), "freq_caveat_ok": freq_ok})
    print(f"  saved {FIG_DIR / (name + '.png')}" + ("" if freq_ok else "  <- MISSING frequency caveat"))


def sha16(*arrays):
    h = hashlib.sha1()
    for a in arrays:
        h.update(np.ascontiguousarray(a).tobytes())
    return h.hexdigest()[:16]

## 1. Inventory and integrity
One pass over the data, then everything is read from a compact cache. **Order:** the metadata audit (**1a**) runs
**before** the full parse (**1b**). The 4 corrupt missions have to be identified from their metadata before they can
be left out of the parse. They are never fully parsed.
- **1a — metadata audit.** This is the ONLY place `SensorCombined` is opened, and only its first and last `timestamp`
  are read, with a regex on the raw line (no IMU value is decoded). The `AudioBuffer` first record gives
  `recording_started_time`.
- **1b — parse + cache.** Each kept mission's `AudioBuffer` is parsed once, de-interleaved and reduced to
  `cache/<mission>.npz`.
- **1c — counts.** Missions, classes and days.
- **1d — six-slot verification** across all 95 missions.

*Cannot show:* the true sampling rate or per-chunk times. Neither exists in the files.

In [ ]:
# ==============================================================================
# 1a. METADATA AUDIT — recording_started_time vs. SensorCombined first/last timestamp; exclusion rule
# ==============================================================================
def mission_day(name: str) -> str:
    d = name[8:18].replace("_", "-")            # rosbag2_YYYY_MM_DD-... -> YYYY-MM-DD
    return DAY_MERGE.get(d, d)


def find_one(mdir: Path, pattern: str):
    hits = sorted(mdir.glob(pattern))
    return hits[0] if hits else None


ALL_DIRS, LABEL_OF_DIR = [], {}
for class_dir in sorted(DATA_ROOT.iterdir()):
    try:
        lab = int(class_dir.name)
    except ValueError:
        continue
    for mdir in sorted(class_dir.iterdir()):
        if mdir.is_dir() and not mdir.name.startswith("."):
            ALL_DIRS.append(mdir)
            LABEL_OF_DIR[mdir] = lab
print(f"Mission folders found: {len(ALL_DIRS)} | per class {np.bincount(list(LABEL_OF_DIR.values()), minlength=N_CLASSES).tolist()}")

_TS_RE = re.compile(rb'"timestamp"\s*:\s*(\d+)')


def sensor_first_last_timestamp(path):
    """First and last `timestamp` (microseconds) of a SensorCombined file: the first line, then a backwards scan of the
    file tail. Regex on raw bytes: no IMU signal value is decoded."""
    with open_logged(path, "SensorCombined:first_last_timestamp", "rb") as f:
        first = None
        for line in f:
            m = _TS_RE.search(line)
            if m:
                first = int(m.group(1))
                break
        f.seek(0, 2)
        size = f.tell()
        block, last = min(size, 1 << 16), None
        while last is None:
            f.seek(size - block)
            lines = f.read(block).splitlines()
            for line in reversed(lines[1:] if block < size else lines):   # lines[0] may be a partial line
                m = _TS_RE.search(line)
                if m:
                    last = int(m.group(1))
                    break
            if last is None and block == size:
                break
            block = min(size, block * 4)
    return first, last


def audio_first_record(path):
    with open_logged(path, "AudioBuffer:first_record") as f:
        for line in f:
            if line.strip():
                return json.loads(line)
    return None


_meta = []
for mdir in ALL_DIRS:
    a_path, s_path = find_one(mdir, "*AudioBuffer*.jsonl"), find_one(mdir, "*SensorCombined*.jsonl")
    assert a_path is not None and s_path is not None, f"{mdir.name}: missing AudioBuffer or SensorCombined file"
    rec = audio_first_record(a_path)
    assert rec is not None and "data" in rec and "recording_started_time" in rec, f"{mdir.name}: unexpected AudioBuffer keys"
    t0, t1 = sensor_first_last_timestamp(s_path)
    rst = int(rec["recording_started_time"])
    dur = (t1 - t0) / 1e6
    off = (rst - t0) / 1e6
    _meta.append({"mission": mdir.name, "class": LABEL_OF_DIR[mdir], "day": mission_day(mdir.name),
                  "recording_started_time": rst, "imu_first_us": t0, "imu_last_us": t1, "audio_offset_s": off,
                  "mission_duration_s": dur, "offset_over_duration": off / dur})
META = pd.DataFrame(_meta).set_index("mission")

EXCLUDED = set(META.index[META.offset_over_duration >= EXCLUDE_RATIO])
print("\nExcluded (offset / duration >= 0.5):")
print(META.loc[sorted(EXCLUDED), ["class", "day", "audio_offset_s", "mission_duration_s", "offset_over_duration"]].to_string())
_kept_meta = META.drop(index=list(EXCLUDED))
print(f"\nMax offset/duration among KEPT missions: {_kept_meta.offset_over_duration.max():.3f} "
      f"({_kept_meta.offset_over_duration.idxmax()}) — below {EXCLUDE_RATIO}")
print(f"Audio start offset among kept missions: median {_kept_meta.audio_offset_s.median():.2f} s, "
      f"range {_kept_meta.audio_offset_s.min():.2f}–{_kept_meta.audio_offset_s.max():.2f} s")
EXCLUSION_OK = (EXCLUDED == EXPECTED_EXCLUDED) and bool((_kept_meta.offset_over_duration < EXCLUDE_RATIO).all())
print(f"Excluded set == the 4 expected missions and no other mission has ratio >= {EXCLUDE_RATIO}: {EXCLUSION_OK}")

fig, axs = plt.subplots(1, 2, figsize=(12, 3.6))
axs[0].hist(_kept_meta.audio_offset_s, bins=30, color=CLASS_COLORS[0], edgecolor="white", label="kept (95)")
axs[0].hist(META.loc[sorted(EXCLUDED), "audio_offset_s"].clip(upper=_kept_meta.audio_offset_s.max() * 3), bins=10,
            color=CLASS_COLORS[1], edgecolor="white", label="excluded (4, clipped)")
axs[0].set_xlabel("audio start offset (s) = (recording_started_time − first IMU timestamp) / 1e6")
axs[0].set_ylabel("missions")
axs[0].legend()
axs[1].scatter(META.mission_duration_s, META.offset_over_duration.clip(-0.2, 2), s=14,
               c=[CLASS_COLORS[1] if n in EXCLUDED else CLASS_COLORS[0] for n in META.index])
axs[1].axhline(EXCLUDE_RATIO, color="#52514e", lw=1, ls="--")
axs[1].set_xlabel("mission duration (s, first->last IMU timestamp)")
axs[1].set_ylabel("offset / duration (clipped to [-0.2, 2])")
fig.suptitle("Audio start offset relative to the IMU log (metadata only)")
save_fig(fig, "1a_audio_offset_distribution",
         "Only timestamps are used: recording_started_time (AudioBuffer) and the first/last SensorCombined timestamp. "
         "Dashed line = exclusion threshold 0.5.")

KEPT = sorted((d for d in ALL_DIRS if d.name not in EXCLUDED), key=lambda d: d.name)
LABEL = {d: LABEL_OF_DIR[d] for d in KEPT}                      # keyed by Path (v27's inner-val rule expects this)
DAY = {d: mission_day(d.name) for d in KEPT}
NAME2DIR = {d.name: d for d in KEPT}
LABEL_N = {d.name: LABEL[d] for d in KEPT}
DAY_N = {d.name: DAY[d] for d in KEPT}
print("\nState: no IMU signal values are read anywhere in this notebook — SensorCombined files are opened only here, "
      "and only their first and last 'timestamp' fields are extracted.")

In [ ]:
# ==============================================================================
# 1b. PARSE EACH KEPT AudioBuffer FILE ONCE -> cache/<mission>.npz (skipped if the cache exists)
# ==============================================================================
def parse_and_cache(mdir: Path) -> str:
    cache = CACHE_DIR / f"{mdir.name}.npz"
    if cache.exists():
        return "cached"
    path = find_one(mdir, "*AudioBuffer*.jsonl")
    chunks, lens, rsts, n_bad = [], Counter(), set(), 0
    with open_logged(path, "AudioBuffer:full_parse") as f:
        for line in f:
            if not line.strip():
                continue
            try:
                rec = json.loads(line)
            except json.JSONDecodeError:
                n_bad += 1
                continue
            x = rec.get("data")
            if x is None:
                n_bad += 1
                continue
            lens[len(x)] += 1
            chunks.append(x)
            rsts.add(rec.get("recording_started_time"))
    modal = lens.most_common(1)[0][0]
    assert modal == BUFFER_LEN, f"{mdir.name}: modal buffer length {modal} != {BUFFER_LEN}"
    assert len(rsts) == 1, f"{mdir.name}: recording_started_time is not constant ({len(rsts)} values)"
    kept = [x for x in chunks if len(x) == modal]
    raw = np.asarray(kept, dtype=np.int64)                      # (n, 1536)
    del chunks, kept
    n = len(raw)
    frames = raw.reshape(n, FRAME, N_SLOTS)                     # sample i -> frame i // 6, slot i % 6
    slot_vals = frames.reshape(-1, N_SLOTS).astype(np.float64)
    ch = frames[:, :, KEEP_SLOTS].astype(np.float64)
    ch = ch - ch.mean(axis=1, keepdims=True)                    # per-chunk mean removed, per channel
    rms = np.sqrt((ch ** 2).mean(axis=1)).astype(np.float32)    # (n, 5)
    pw = (np.abs(np.fft.rfft(ch, axis=1)) ** 2).transpose(0, 2, 1).astype(np.float32)   # (n, 5, 129), rectangular window
    vmin, vmax = int(raw.min()), int(raw.max())
    tmp = CACHE_DIR / f"{mdir.name}.tmp.npz"
    np.savez(tmp, pow=pw, rms=rms, slot_std=slot_vals.std(axis=0), slot_mean=slot_vals.mean(axis=0),
             slot_zero=np.all(slot_vals == 0, axis=0), slot_absmax=np.abs(slot_vals).max(axis=0), n_chunks=n,
             mid_buffer=raw[n // 2].astype(np.int32), vmin=vmin, vmax=vmax, n_at_min=int((raw == vmin).sum()),
             n_at_max=int((raw == vmax).sum()), n_dropped=sum(v for k, v in lens.items() if k != modal),
             n_bad_lines=n_bad, modal_len=modal, recording_started_time=int(next(iter(rsts))))
    os.replace(tmp, cache)                                      # atomic: an interrupted parse never leaves a bad cache
    return "parsed"


_t0 = time.time()
PARSE_STATUS = {}
for i, mdir in enumerate(KEPT, 1):
    PARSE_STATUS[mdir.name] = parse_and_cache(mdir)
    if i % 10 == 0 or i == len(KEPT):
        print(f"  {i}/{len(KEPT)} missions | parsed {sum(v == 'parsed' for v in PARSE_STATUS.values())}, cached "
              f"{sum(v == 'cached' for v in PARSE_STATUS.values())} | {time.time() - _t0:.0f} s")

# ---- ONE load pass over the cache: keep only the compact derived arrays used below ----
MISS = {}
for mdir in KEPT:
    z = np.load(CACHE_DIR / f"{mdir.name}.npz")
    pw = z["pow"]
    n = len(pw)
    a, b = int(STEADY[0] * n), int(STEADY[1] * n)
    MISS[mdir.name] = {
        "n_chunks": n, "rms": z["rms"], "steady_pow": pw[a:b].astype(np.float64).mean(axis=0),   # (5, 129)
        "feats": (pw[:, :, KEEP_BINS] ** 0.25).reshape(n, N_FEAT).astype(np.float32),        # sqrt|rFFT|, channel-major
        "band": pw[:, :, KEEP_BINS].sum(axis=2),                                               # (n, 5) power in bins 1-16
        "slot_std": z["slot_std"], "slot_mean": z["slot_mean"], "slot_zero": z["slot_zero"], "mid_buffer": z["mid_buffer"],
        "vmin": int(z["vmin"]), "vmax": int(z["vmax"]), "n_at_min": int(z["n_at_min"]), "n_at_max": int(z["n_at_max"]),
        "n_dropped": int(z["n_dropped"]), "n_bad_lines": int(z["n_bad_lines"]), "modal_len": int(z["modal_len"]),
        "finite": bool(np.isfinite(pw).all()), "rst_cache": int(z["recording_started_time"])}
    del pw, z
_dropped = sum(v["n_dropped"] for v in MISS.values())
print(f"\nModal buffer length = {BUFFER_LEN} in every mission: {all(v['modal_len'] == BUFFER_LEN for v in MISS.values())} | "
      f"non-modal chunks dropped: {_dropped} (in {sum(v['n_dropped'] > 0 for v in MISS.values())} missions) | "
      f"undecodable lines: {sum(v['n_bad_lines'] for v in MISS.values())}")
assert all(v["modal_len"] == BUFFER_LEN for v in MISS.values())
assert all(v["rst_cache"] == META.loc[k, "recording_started_time"] for k, v in MISS.items()), "cache/metadata mismatch"
print(f"Cache: {len(MISS)} missions, {sum(v['n_chunks'] for v in MISS.values()):,} chunks | load pass {time.time() - _t0:.0f} s total")

In [ ]:
# ==============================================================================
# 1c. COUNTS — missions, classes, days (HOOK 1)
# ==============================================================================
_cls = np.bincount([LABEL[d] for d in KEPT], minlength=N_CLASSES).tolist()
_days = [sum(DAY[d] == day for d in KEPT) for day in DAYS]
DAY_CLASS = pd.crosstab(pd.Series([DAY[d] for d in KEPT], name="day"), pd.Series([LABEL[d] for d in KEPT], name="class"))
DAY_CLASS = DAY_CLASS.reindex(index=DAYS, columns=range(N_CLASSES), fill_value=0)
print(f"Missions: {len(KEPT)} | per class {_cls} | per day {dict(zip(DAYS, _days))}")
print(f"Merged into 2021-12-31: {[d.name for d in KEPT if d.name[8:18] == '2021_12_29']}")
print("\nDay x class (missions):")
print(DAY_CLASS.assign(total=DAY_CLASS.sum(axis=1)).to_string())
print(f"Every class on every day: {bool((DAY_CLASS.values > 0).all())}")
hook(1, len(KEPT) == EXPECTED_N_MISSIONS and _cls == EXPECTED_CLASS_COUNTS and EXCLUSION_OK and _days == EXPECTED_DAY_COUNTS
     and bool((DAY_CLASS.values > 0).all()),
     f"{len(KEPT)} missions, classes {_cls}, excluded set == the 4 names: {EXCLUSION_OK}, days {_days} "
     f"(expected {EXPECTED_N_MISSIONS}, {EXPECTED_CLASS_COUNTS}, True, {EXPECTED_DAY_COUNTS})")

In [ ]:
# ==============================================================================
# 1d. SIX-SLOT VERIFICATION across all 95 missions (HOOK 2)
# ==============================================================================
_std = np.stack([MISS[d.name]["slot_std"] for d in KEPT])        # (95, 6)
_mean = np.stack([MISS[d.name]["slot_mean"] for d in KEPT])
_zero = np.stack([MISS[d.name]["slot_zero"] for d in KEPT])
_slot_tab = pd.DataFrame({"slot": [f"slot {s + 1}" for s in range(N_SLOTS)], "mean std": _std.mean(0).round(2),
                          "median std": np.median(_std, 0).round(2), "mean DC (slot mean)": _mean.mean(0).round(2),
                          "missions identically zero": _zero.sum(0),
                          "missions where largest std": np.bincount(_std.argmax(1), minlength=N_SLOTS)}).set_index("slot")
print("Per-slot statistics over all samples of each mission, then across the 95 missions:")
print(_slot_tab.to_string())
n_zero6 = int(_zero[:, DROP_SLOT].sum())
n_slot1_max = int((_std.argmax(1) == 0).sum())
print(f"\nSlot 6 identically zero in {n_zero6}/{len(KEPT)} missions | slot 1 has the largest std in {n_slot1_max}/{len(KEPT)}")

_vmin = min(v["vmin"] for v in MISS.values())
_vmax = max(v["vmax"] for v in MISS.values())
_at_ext = [(k, v["n_at_min"], v["n_at_max"]) for k, v in MISS.items() if v["vmin"] == _vmin or v["vmax"] == _vmax]
print(f"\nClipping/saturation: observed global sample range [{_vmin}, {_vmax}] "
      f"(int16 full scale is [-32768, 32767]: {'reached' if _vmin <= -32768 or _vmax >= 32767 else 'NOT reached'})")
print(f"  missions touching the global extreme: {len(_at_ext)}; samples sitting exactly at their own min/max per mission "
      f"(median over missions): {np.median([v['n_at_min'] + v['n_at_max'] for v in MISS.values()]):.0f} "
      f"(a large count at the extreme would indicate clipping)")
print(f"NaN/Inf in any cached spectrum: {not all(v['finite'] for v in MISS.values())}")

_nch = pd.DataFrame({"mission": [d.name for d in KEPT], "class": [LABEL[d] for d in KEPT], "day": [DAY[d] for d in KEPT],
                     "chunks": [MISS[d.name]["n_chunks"] for d in KEPT]})
print("\nChunks per mission: " + _nch.chunks.describe()[["min", "25%", "50%", "75%", "max"]].round(0).to_string().replace("\n", " | "))
print("Chunks per class:")
print(_nch.groupby("class").chunks.agg(["count", "sum", "median", "min", "max"]).to_string())
_imp_ms = [(META.loc[d.name, "imu_last_us"] - META.loc[d.name, "recording_started_time"]) / 1e3 / MISS[d.name]["n_chunks"]
           for d in KEPT]
print(f"\nDESCRIPTIVE ONLY (not used anywhere): if audio ran from recording_started_time to the last IMU timestamp, one "
      f"buffer would last {np.percentile(_imp_ms, 5):.1f}–{np.percentile(_imp_ms, 95):.1f} ms (5th–95th pct, median "
      f"{np.median(_imp_ms):.1f} ms). 16 ms (16 kHz per channel) is consistent with this, but it is unverified. A 48 kHz "
      f"mono assumption would imply {BUFFER_LEN / 48:.0f} ms.")
hook(2, n_zero6 == len(KEPT) == EXPECTED_N_MISSIONS, f"slot 6 identically zero in {n_zero6}/{len(KEPT)} missions")

## 2. Visual tour (de-interleaved channels; every figure saved to `figures/` as PNG)
Every frequency axis is a **per-channel FFT bin (1–128)**. A secondary axis gives approximate Hz *if* the per-channel
rate were 16 kHz. That rate is unverified. No figure here shows a mono FFT of the interleaved buffer: that FFT has
spectral images at multiples of bin 256 (N/6), not physical frequencies.

**2a — structure.** The first 60 interleaved samples of one mid-flight buffer, beside the 6 slots de-interleaved
over 256 frames.
*Cannot show:* which microphone each slot is, or the sample rate.

In [ ]:
# ==============================================================================
# 2a. STRUCTURE — interleaved sawtooth vs. de-interleaved channels
# ==============================================================================
EX_MISSION = sorted((d for d in KEPT if LABEL[d] == 0), key=lambda d: d.name)[0].name
_buf = MISS[EX_MISSION]["mid_buffer"].astype(np.float64)
_fr = _buf.reshape(FRAME, N_SLOTS)
fig = plt.figure(figsize=(13, 8))
gs = fig.add_gridspec(N_SLOTS, 2, width_ratios=[1.1, 1])
ax0 = fig.add_subplot(gs[:, 0])
ax0.plot(np.arange(60), _buf[:60], color="#9a9994", lw=1)
for s in range(N_SLOTS):
    idx = np.arange(s, 60, N_SLOTS)
    ax0.scatter(idx, _buf[idx], s=22, color=(CLASS_COLORS + ["#52514e"])[s], label=f"slot {s + 1}", zorder=3)
ax0.set_xlabel("interleaved sample index in the buffer (0-59)")
ax0.set_ylabel("raw sample value")
ax0.set_title(f"First 60 interleaved samples — {EX_MISSION}, mid-flight buffer")
ax0.legend(ncol=3)
for s in range(N_SLOTS):
    ax = fig.add_subplot(gs[s, 1])
    ax.plot(_fr[:, s], color=(CLASS_COLORS + ["#52514e"])[s], lw=0.9)
    ax.set_ylabel(f"slot {s + 1}" + (" (empty)" if s == DROP_SLOT else f"\n= ch {KEEP_SLOTS.index(s) + 1}"), fontsize=8)
    if s < N_SLOTS - 1:
        ax.set_xticklabels([])
ax.set_xlabel("frame (0-255) within the buffer")
fig.suptitle("One 1536-sample buffer = 256 frames x 6 interleaved slots")
save_fig(fig, "2a_buffer_structure",
         "Left: the raw interleaved order. Its sawtooth comes from the different per-slot DC levels. Right: the same "
         "buffer de-interleaved (sample i -> frame i//6, slot i%6); slot 6 is identically zero. The time axes are sample "
         "and frame indices, not seconds (the sample rate is unknown).")

**2b — loudness over the flight.** Per-chunk RMS, the mean over the 5 channels after per-chunk mean removal. The
x-axis is **relative chunk position (0–100 % of the mission's chunks), i.e. chunk order, not seconds.** The heatmap
shows the start (spin-up), steady flight and the landing decay for all 95 missions.
*Cannot show:* absolute time, or whether the gain was identical across days.

In [ ]:
# ==============================================================================
# 2b. LOUDNESS OVER THE FLIGHT — one mission per class, then all 95 missions
# ==============================================================================
def mean_rms(name):
    return MISS[name]["rms"].astype(np.float64).mean(axis=1)


def resample_bins(x, n_out=100):
    edges = np.linspace(0, len(x), n_out + 1).astype(int)
    return np.array([x[edges[i]:max(edges[i + 1], edges[i] + 1)].mean() for i in range(n_out)])


# one representative (median-length) mission per class
REP = {}
for c in range(N_CLASSES):
    ms = sorted((d.name for d in KEPT if LABEL[d] == c), key=lambda n: MISS[n]["n_chunks"])
    REP[c] = ms[len(ms) // 2]

fig, axs = plt.subplots(1, 2, figsize=(14, 4.2), gridspec_kw={"width_ratios": [1.2, 1]})
for c in range(N_CLASSES):
    r = mean_rms(REP[c])
    k = max(1, len(r) // 200)
    sm = np.convolve(r, np.ones(k) / k, mode="same")
    axs[0].plot(np.linspace(0, 100, len(r)), sm, color=CLASS_COLORS[c], label=f"class {c} ({DAY_N[REP[c]]})")
axs[0].axvspan(0, 100 * STEADY[0], color="#d8d7d2", alpha=0.4, lw=0)
axs[0].axvspan(100 * STEADY[1], 100, color="#d8d7d2", alpha=0.4, lw=0)
axs[0].set_xlabel("relative position in the mission (% of chunks) — chunk order, NOT seconds")
axs[0].set_ylabel("RMS (mean of 5 channels, raw units)")
axs[0].set_title("Loudness over the flight, one mission per class (moving average)")
axs[0].legend()

HEAT_ORDER = sorted((d.name for d in KEPT), key=lambda n: (LABEL_N[n], DAYS.index(DAY_N[n]), n))
H = np.stack([np.log10(resample_bins(mean_rms(n)) + 1e-9) for n in HEAT_ORDER])
im = axs[1].imshow(H, aspect="auto", cmap=SEQ_CMAP, extent=[0, 100, len(H), 0], interpolation="nearest")
_bounds = np.cumsum([sum(LABEL_N[n] == c for n in HEAT_ORDER) for c in range(N_CLASSES)])
for y in _bounds[:-1]:
    axs[1].axhline(y, color="#0b0b0b", lw=0.8)
axs[1].set_yticks([(a + b) / 2 for a, b in zip(np.r_[0, _bounds[:-1]], _bounds)])
axs[1].set_yticklabels([f"class {c}" for c in range(N_CLASSES)])
axs[1].set_xlabel("relative position (% of chunks, 100 bins) — chunk order, NOT seconds")
axs[1].set_title("log10 RMS, all 95 missions (rows: class, then day)")
axs[1].grid(False)
fig.colorbar(im, ax=axs[1], label="log10 RMS")
save_fig(fig, "2b_loudness_over_flight",
         "x = relative chunk position (chunk order, not seconds). Grey bands = the first 10% and last 10% of chunks "
         "(start/end phases, excluded from the steady-flight statistics).")
_ratio_start = [mean_rms(n)[:max(1, int(0.02 * MISS[n]['n_chunks']))].mean() /
                mean_rms(n)[int(STEADY[0] * MISS[n]['n_chunks']):int(STEADY[1] * MISS[n]['n_chunks'])].mean() for n in HEAT_ORDER]
print(f"RMS of the first 2% of chunks / steady-flight RMS: median {np.median(_ratio_start):.2f} "
      f"(IQR {np.percentile(_ratio_start, 25):.2f}–{np.percentile(_ratio_start, 75):.2f}) — spin-up is much quieter than flight")

**2c — spectrograms.** Channels 1 and 4, one steady-flight segment per class: 512 chunks centred at 50 % of the
mission. Time = chunk index; y = per-channel bin 1–128; colour = log10 power.
*Cannot show:* class-level behaviour. Each panel is one mission.

In [ ]:
# ==============================================================================
# 2c. SPECTROGRAMS — channels 1 and 4, one steady-flight segment per class
# ==============================================================================
SPEC_CH = [0, 3]
SEG = 512
_segs = {}
for c in range(N_CLASSES):
    pw = np.load(CACHE_DIR / f"{REP[c]}.npz")["pow"]
    s0 = max(0, len(pw) // 2 - SEG // 2)
    _segs[c] = (s0, np.log10(pw[s0:s0 + SEG, :, 1:N_RFFT].astype(np.float64) + 1e-3))
    del pw
fig, axs = plt.subplots(len(SPEC_CH), N_CLASSES, figsize=(18, 6.8), sharey=True)
freq_axes = []
for r, ch in enumerate(SPEC_CH):
    vals = np.concatenate([_segs[c][1][:, ch, :].ravel() for c in range(N_CLASSES)])
    vmin, vmax = np.percentile(vals, [2, 99.5])
    for c in range(N_CLASSES):
        s0, L = _segs[c]
        ax = axs[r, c]
        im = ax.imshow(L[:, ch, :].T, origin="lower", aspect="auto", cmap=SEQ_CMAP, vmin=vmin, vmax=vmax,
                       extent=[s0, s0 + L.shape[0], 0.5, N_RFFT - 0.5], interpolation="nearest")
        ax.grid(False)
        ax.set_title(f"class {c} — ch {ch + 1}", fontsize=9)
        if c == 0:
            ax.set_ylabel(FREQ_LABEL)
        if c == N_CLASSES - 1:
            add_bin_axis(ax, "y")
        else:
            ax._bin_caveat = True          # shares its y-axis with the last panel of the row (which carries the Hz axis)
        freq_axes.append(ax)
        if r == len(SPEC_CH) - 1:
            ax.set_xlabel("chunk index")
    fig.colorbar(im, ax=axs[r, :].tolist(), label="log10 power", fraction=0.015)
fig.suptitle("Steady-flight spectrograms (512 chunks at mid-mission), de-interleaved channels 1 and 4")
save_fig(fig, "2c_spectrograms_ch1_ch4", FREQ_CAVEAT + " x = chunk index (not seconds). Colour scale shared within a row.",
         freq_axes=freq_axes)
del _segs

**2d — class-conditional mean log-power spectra** per channel. Steady flight only (10–90 % of chunks). One value per
mission, then the class mean with a ±1 mission-level standard-error band. Absolute level, **not normalized**:
loudness is part of the signal.
*Cannot show:* whether a difference holds within each day (see 2f and section 3b).

In [ ]:
# ==============================================================================
# 2d. CLASS-CONDITIONAL MEAN LOG-POWER SPECTRA (steady flight, mission-level mean ± SE)
# ==============================================================================
NAMES = [d.name for d in KEPT]
Y_M = np.array([LABEL_N[n] for n in NAMES])
DAY_M = np.array([DAY_N[n] for n in NAMES])
LOGP = np.stack([np.log10(MISS[n]["steady_pow"][:, 1:N_RFFT] + 1e-12) for n in NAMES])      # (95, 5, 128): bins 1..128
BINS = np.arange(1, N_RFFT)

fig, axs = plt.subplots(2, 3, figsize=(16, 8))
freq_axes = []
for ch in range(N_CH):
    ax = axs.flat[ch]
    for c in range(N_CLASSES):
        X = LOGP[Y_M == c, ch, :]
        m, se = X.mean(0), X.std(0, ddof=1) / np.sqrt(len(X))
        ax.plot(BINS, m, color=CLASS_COLORS[c], lw=1.3, label=f"class {c} (n={len(X)})")
        ax.fill_between(BINS, m - se, m + se, color=CLASS_COLORS[c], alpha=0.18, lw=0)
    ax.set_title(f"channel {ch + 1}")
    ax.set_xlabel(FREQ_LABEL)
    ax.set_ylabel("log10 mean power (steady flight)")
    add_bin_axis(ax)
    freq_axes.append(ax)
axs.flat[0].legend()
axs.flat[-1].axis("off")
fig.suptitle("Class-mean log-power spectra per channel (mission-level mean ± 1 SE), absolute level")
save_fig(fig, "2d_class_mean_spectra", FREQ_CAVEAT + " Band = ±1 standard error across missions (n = 18–20 per class).",
         freq_axes=freq_axes)

**2e — spatial signature.** Per-channel steady-flight RMS by class, with mission-level 95 % t-intervals and one dot per
mission. Then the between-channel correlation of mission-level log-RMS.
*Cannot show:* the microphone geometry. It is not in the files.

In [ ]:
# ==============================================================================
# 2e. SPATIAL SIGNATURE — per-channel RMS by class; between-channel correlation of mission log-RMS
# ==============================================================================
from scipy import stats

def steady_slice(n):
    N = MISS[n]["n_chunks"]
    return slice(int(STEADY[0] * N), int(STEADY[1] * N))


RMS_M = np.stack([MISS[n]["rms"][steady_slice(n)].astype(np.float64).mean(0) for n in NAMES])   # (95, 5)
LOGRMS = np.log10(RMS_M)
fig, axs = plt.subplots(1, N_CH + 1, figsize=(19, 3.8), gridspec_kw={"width_ratios": [1] * N_CH + [1.25]})
_rng = np.random.default_rng(0)
for ch in range(N_CH):
    ax = axs[ch]
    for c in range(N_CLASSES):
        v = RMS_M[Y_M == c, ch]
        h = stats.t.ppf(0.975, len(v) - 1) * v.std(ddof=1) / np.sqrt(len(v))
        ax.scatter(c + _rng.uniform(-0.18, 0.18, len(v)), v, s=9, color=CLASS_COLORS[c], alpha=0.5, lw=0)
        ax.errorbar(c, v.mean(), yerr=h, fmt="o", color="#0b0b0b", ms=5, capsize=3, lw=1.2)
    ax.set_xticks(range(N_CLASSES))
    ax.set_xlabel("class")
    ax.set_title(f"channel {ch + 1}")
    if ch == 0:
        ax.set_ylabel("steady-flight RMS (raw units)")
C = np.corrcoef(LOGRMS.T)
im = axs[-1].imshow(C, cmap=DIV_CMAP, vmin=-1, vmax=1)
axs[-1].grid(False)
for i in range(N_CH):
    for j in range(N_CH):
        axs[-1].text(j, i, f"{C[i, j]:.2f}", ha="center", va="center", fontsize=7, color="#0b0b0b")
axs[-1].set_xticks(range(N_CH))
axs[-1].set_xticklabels([f"ch{i + 1}" for i in range(N_CH)])
axs[-1].set_yticks(range(N_CH))
axs[-1].set_yticklabels([f"ch{i + 1}" for i in range(N_CH)])
axs[-1].set_title("corr. of mission log-RMS")
fig.colorbar(im, ax=axs[-1], fraction=0.046)
fig.suptitle("Spatial signature: per-channel loudness by class (black = mean ± 95% t-interval; dots = missions)")
save_fig(fig, "2e_spatial_signature",
         "Steady flight only (10–90% of chunks). One value per mission; the interval is across missions, not windows.")
_within = LOGRMS - np.stack([LOGRMS[Y_M == c].mean(0) for c in Y_M])
print("Between-channel correlation of mission log-RMS AFTER removing class means (within-class structure):")
print(pd.DataFrame(np.corrcoef(_within.T), index=[f"ch{i + 1}" for i in range(N_CH)],
                   columns=[f"ch{i + 1}" for i in range(N_CH)]).round(2).to_string())

**2f — day effect.** The mean spectrum per recording day for class 0 and class 4 (channels 1 and 4). Then a PCA of
standardized mission-level log-power (5 channels × bins 1–32 = 160 features), plotted twice: coloured by class and by
day. The printed shares are the fraction of total standardized feature variance explained by class, and by day within
class, against the chance level (df / (n − 1)).
*Cannot show:* the cause of a day effect (gain, weather, battery, microphone placement).

In [ ]:
# ==============================================================================
# 2f. DAY EFFECT — per-day spectra; PCA by class and by day; variance shares vs. chance
# ==============================================================================
fig, axs = plt.subplots(2, 2, figsize=(14, 8), sharex=True)
freq_axes = []
for r, c in enumerate([0, 4]):
    for k, ch in enumerate([0, 3]):
        ax = axs[r, k]
        for di, day in enumerate(DAYS):
            sel = (Y_M == c) & (DAY_M == day)
            if sel.sum() == 0:
                continue
            ax.plot(BINS, LOGP[sel, ch, :].mean(0), color=DAY_COLORS[di], lw=1.2, label=f"{day} (n={sel.sum()})")
        ax.set_title(f"class {c} — channel {ch + 1}")
        ax.set_ylabel("log10 mean power")
        ax.set_xlabel(FREQ_LABEL)
        add_bin_axis(ax)
        freq_axes.append(ax)
        ax.legend(fontsize=7)
fig.suptitle("Mean steady-flight spectrum per recording day (same class, different days)")
save_fig(fig, "2f_day_spectra_class0_class4", FREQ_CAVEAT + " Each line = mean over that day's missions of the class.",
         freq_axes=freq_axes)

PCA_X = LOGP[:, :, np.array(PCA_BINS) - 1].reshape(len(NAMES), -1)                 # (95, 160)
Z = (PCA_X - PCA_X.mean(0)) / PCA_X.std(0, ddof=0)
U, S, Vt = np.linalg.svd(Z, full_matrices=False)
PC = U[:, :2] * S[:2]
EVR = S ** 2 / (S ** 2).sum()
fig, axs = plt.subplots(1, 2, figsize=(13, 5))
for c in range(N_CLASSES):
    s = Y_M == c
    axs[0].scatter(PC[s, 0], PC[s, 1], s=30, color=CLASS_COLORS[c], marker=CLASS_MARKERS[c], label=f"class {c}",
                   edgecolor="white", lw=0.6)
for di, day in enumerate(DAYS):
    s = DAY_M == day
    axs[1].scatter(PC[s, 0], PC[s, 1], s=30, color=DAY_COLORS[di], marker=DAY_MARKERS[di], label=day, edgecolor="white", lw=0.6)
for ax, t in zip(axs, ["coloured by class", "coloured by recording day"]):
    ax.set_xlabel(f"PC1 ({100 * EVR[0]:.1f}% of variance)")
    ax.set_ylabel(f"PC2 ({100 * EVR[1]:.1f}%)")
    ax.set_title(t)
    ax.legend()
fig.suptitle("PCA of standardized mission-level log-power (5 channels x bins 1-32 = 160 features, 95 missions)")
save_fig(fig, "2f_pca_class_vs_day", "Each point is one mission (steady flight). PCA is unsupervised; neither label is used "
                                     "to fit it. Colour and marker encode the same label.")


def variance_shares(Zs, y, days):
    """Share of total standardized feature variance explained by class, and by day within class (nested)."""
    n = len(y)
    tot = ((Zs - Zs.mean(0)) ** 2).sum()
    ss_class = sum((y == c).sum() * ((Zs[y == c].mean(0) - Zs.mean(0)) ** 2).sum() for c in np.unique(y))
    ss_day_in_class, df_day = 0.0, 0
    for c in np.unique(y):
        mc = Zs[y == c].mean(0)
        ds = np.unique(days[y == c])
        df_day += len(ds) - 1
        for d in ds:
            s = (y == c) & (days == d)
            ss_day_in_class += s.sum() * ((Zs[s].mean(0) - mc) ** 2).sum()
    ss_day = sum((days == d).sum() * ((Zs[days == d].mean(0) - Zs.mean(0)) ** 2).sum() for d in np.unique(days))
    return {"class": (ss_class / tot, (len(np.unique(y)) - 1) / (n - 1)),
            "day (ignoring class)": (ss_day / tot, (len(np.unique(days)) - 1) / (n - 1)),
            "day within class": (ss_day_in_class / tot, df_day / (n - 1))}


VAR_SHARES = variance_shares(Z, Y_M, DAY_M)
print("Share of standardized feature variance (160 features, 95 missions) explained by:")
for k, (share, chance) in VAR_SHARES.items():
    print(f"  {k:<22} {share:.3f}   chance level (df/(n-1)) {chance:.3f}   ratio {share / chance:.1f}x")

## 3. Where the classes differ (mission-level, all 95 missions, DESCRIPTIVE ONLY)
> **Note.** This section uses all 95 missions, test missions included. It is **not** used to select model features:
> the model's input (section 4) was fixed in advance. It is **not a performance claim** either: an effect size is not
> a classification accuracy.

Feature per (channel, bin 1–128) = log10 of the mission's mean steady-flight power (the `LOGP` array from 2d).
- **3a** — one-way ANOVA F per cell. The family-wise threshold is the 95th percentile of the max-F null, built from
  1000 permutations of class labels **within each day**.
- **3b** — Cohen's d for the adjacent pairs, the top-10 cells per pair with bootstrap CIs, and day-sign consistency.
- **3c** — monotonicity (Spearman ρ of class index with each cell).
- **3d** — within-mission stability: a variance decomposition of window-level band power.

In [ ]:
# ==============================================================================
# 3a. ONE-WAY ANOVA F per (channel, bin), family-wise threshold from within-day label permutations (max-F null)
# ==============================================================================
XF = LOGP.reshape(len(NAMES), -1)                         # (95, 640): channel-major, bins 1..128


def anova_F(X, y, K=N_CLASSES):
    grand = X.mean(0)
    ssb = np.zeros(X.shape[1])
    ssw = np.zeros(X.shape[1])
    for c in range(K):
        Xc = X[y == c]
        mc = Xc.mean(0)
        ssb += len(Xc) * (mc - grand) ** 2
        ssw += ((Xc - mc) ** 2).sum(0)
    return (ssb / (K - 1)) / (ssw / (len(y) - K))


F_OBS = anova_F(XF, Y_M)
_rng = np.random.default_rng(PERM_SEED)
_day_idx = [np.where(DAY_M == d)[0] for d in DAYS]
MAXF_NULL = np.empty(N_PERMUTATIONS)
for p in range(N_PERMUTATIONS):
    yp = Y_M.copy()
    for idx in _day_idx:
        yp[idx] = Y_M[_rng.permutation(idx)]
    MAXF_NULL[p] = anova_F(XF, yp).max()
F_THR = float(np.percentile(MAXF_NULL, 95))
SIG = F_OBS > F_THR
print(f"Max-F null ({N_PERMUTATIONS} within-day permutations): 95th percentile = {F_THR:.2f} | observed max F = {F_OBS.max():.2f}")
print(f"Cells above the family-wise threshold: {SIG.sum()} / {SIG.size} | per channel: "
      + ", ".join(f"ch{c + 1} {SIG.reshape(N_CH, -1)[c].sum()}" for c in range(N_CH)))
print(f"  of which in the model's bins 1-16: {SIG.reshape(N_CH, -1)[:, :16].sum()} (descriptive — the model's bins were fixed a priori)")

fig, ax = plt.subplots(figsize=(15, 3.4))
Fm = F_OBS.reshape(N_CH, -1)
im = ax.imshow(np.log10(Fm), aspect="auto", cmap=SEQ_CMAP, extent=[0.5, 128.5, N_CH + 0.5, 0.5], interpolation="nearest")
yy, xx = np.where(SIG.reshape(N_CH, -1))
ax.scatter(xx + 1, yy + 1, s=4, color="#e34948", marker="s", label=f"F > family-wise threshold ({F_THR:.1f})")
ax.set_yticks(range(1, N_CH + 1))
ax.set_yticklabels([f"ch{c}" for c in range(1, N_CH + 1)])
ax.set_xlabel(FREQ_LABEL)
ax.grid(False)
add_bin_axis(ax)
ax.legend(loc="lower right")
fig.colorbar(im, ax=ax, label="log10 F", fraction=0.02)
ax.set_title("One-way ANOVA F (class) per channel x bin, mission-level steady-flight log-power")
save_fig(fig, "3a_anova_F", FREQ_CAVEAT + " Descriptive, all 95 missions; not used to select model inputs. Threshold = 95th "
                            "pct. of max-F over 1000 within-day label permutations.", freq_axes=[ax])

In [ ]:
# ==============================================================================
# 3b. COHEN'S d for adjacent pairs: heatmaps; top-10 cells with bootstrap CI and day-sign consistency
# ==============================================================================
def cohen_d(Xa, Xb):
    na, nb = len(Xa), len(Xb)
    sp = np.sqrt(((na - 1) * Xa.var(0, ddof=1) + (nb - 1) * Xb.var(0, ddof=1)) / (na + nb - 2))
    return (Xb.mean(0) - Xa.mean(0)) / np.where(sp > 0, sp, np.nan)


def cell_name(j):
    return f"ch{j // 128 + 1} bin {j % 128 + 1}"


_boot_rng = np.random.default_rng(BOOTSTRAP_SEED)


def cohen_top_table(a, b, top=TOP_CELLS):
    """Top cells by |d| (b minus a): d, 95% mission-level bootstrap CI (resampling missions within each class), and the
    number of days (with >= 2 missions of both classes) on which the class-mean difference has the same sign as d."""
    Xa, Xb = XF[Y_M == a], XF[Y_M == b]
    d = cohen_d(Xa, Xb)
    order = np.argsort(-np.abs(np.nan_to_num(d)))[:top]
    ia = _boot_rng.integers(0, len(Xa), (BOOTSTRAP_RESAMPLES, len(Xa)))
    ib = _boot_rng.integers(0, len(Xb), (BOOTSTRAP_RESAMPLES, len(Xb)))
    A, B = Xa[:, order][ia], Xb[:, order][ib]             # (B, n, top)
    na, nb = len(Xa), len(Xb)
    sp = np.sqrt(((na - 1) * A.var(1, ddof=1) + (nb - 1) * B.var(1, ddof=1)) / (na + nb - 2))
    dboot = (B.mean(1) - A.mean(1)) / sp
    lo, hi = np.nanpercentile(dboot, [2.5, 97.5], axis=0)
    rows = []
    for k, j in enumerate(order):
        same, elig = 0, 0
        for day in DAYS:
            sa, sb = (Y_M == a) & (DAY_M == day), (Y_M == b) & (DAY_M == day)
            if sa.sum() >= MIN_PER_CLASS_FOR_DAY_SIGN and sb.sum() >= MIN_PER_CLASS_FOR_DAY_SIGN:
                elig += 1
                same += int(np.sign(XF[sb, j].mean() - XF[sa, j].mean()) == np.sign(d[j]))
        rows.append({"cell": cell_name(j), "d": round(float(d[j]), 2), "CI low": round(float(lo[k]), 2),
                     "CI high": round(float(hi[k]), 2), "days same sign": f"{same}/{elig}",
                     "consistent (>=4 days)": same >= SIGN_CONSISTENT_MIN_DAYS})
    return d, pd.DataFrame(rows)


ADJ_PAIRS = [(0, 1), (1, 2), (2, 3), (3, 4)]
D_MAPS, D_TABLES = {}, {}
fig, axs = plt.subplots(len(ADJ_PAIRS), 1, figsize=(15, 11), sharex=True)
freq_axes = []
for ax, (a, b) in zip(axs, ADJ_PAIRS):
    d, tab = cohen_top_table(a, b)
    D_MAPS[(a, b)], D_TABLES[(a, b)] = d, tab
    lim = np.nanpercentile(np.abs(d), 99)
    im = ax.imshow(d.reshape(N_CH, -1), aspect="auto", cmap=DIV_CMAP, vmin=-lim, vmax=lim,
                   extent=[0.5, 128.5, N_CH + 0.5, 0.5], interpolation="nearest")
    ax.set_yticks(range(1, N_CH + 1))
    ax.set_yticklabels([f"ch{c}" for c in range(1, N_CH + 1)])
    ax.grid(False)
    ax.set_title(f"Cohen's d, class {b} minus class {a} (positive = louder in class {b})")
    fig.colorbar(im, ax=ax, fraction=0.02, label="d")
    if ax is axs[0]:
        add_bin_axis(ax)
    else:
        ax._bin_caveat = True              # shared x-axis: the Hz axis is on the top panel
    freq_axes.append(ax)
axs[-1].set_xlabel(FREQ_LABEL)
save_fig(fig, "3b_cohens_d_adjacent_pairs", FREQ_CAVEAT + " Descriptive, all 95 missions; mission-level steady-flight "
                                                          "log-power; the panels share the x-axis (Hz axis on top).",
         freq_axes=freq_axes)
for (a, b), tab in D_TABLES.items():
    print(f"\n=== Section 3b table: class {a} v {b} — top {TOP_CELLS} cells by |d| (d = class {b} − class {a}) ===")
    print(tab.to_string(index=False))
    print(f"  cells consistent on >= {SIGN_CONSISTENT_MIN_DAYS} days: {int(tab['consistent (>=4 days)'].sum())}/{len(tab)}")

In [ ]:
# ==============================================================================
# 3c. MONOTONICITY — Spearman rho of class index with each cell
# ==============================================================================
_ry = stats.rankdata(Y_M)
_rx = stats.rankdata(XF, axis=0)
RHO = np.array([np.corrcoef(_ry, _rx[:, j])[0, 1] for j in range(XF.shape[1])])
_strong = np.abs(RHO) >= SPEARMAN_THRESHOLD
print(f"Cells with |rho| >= {SPEARMAN_THRESHOLD}: {_strong.sum()} / {len(RHO)} "
      f"(positive: {(RHO >= SPEARMAN_THRESHOLD).sum()}, negative: {(RHO <= -SPEARMAN_THRESHOLD).sum()})")
print("  per channel: " + ", ".join(f"ch{c + 1} {_strong.reshape(N_CH, -1)[c].sum()}" for c in range(N_CH)))
print(f"  max |rho| = {np.abs(RHO).max():.2f} at {cell_name(int(np.abs(RHO).argmax()))}")
if _strong.any():
    top = np.argsort(-np.abs(RHO))[:10]
    print("  top 10: " + ", ".join(f"{cell_name(j)} {RHO[j]:+.2f}" for j in top))

In [ ]:
# ==============================================================================
# 3d. WITHIN-MISSION STABILITY — variance of window-level band power (bins 1-16) by class / mission / residual
# ==============================================================================
_rows = []
for n in NAMES:
    band = MISS[n]["band"].astype(np.float64)
    N = len(band)
    for s in range(0, N - WINDOW_CHUNKS + 1, STRIDE_CHUNKS):
        rel = (s + WINDOW_CHUNKS / 2) / N
        if STEADY[0] <= rel <= STEADY[1]:
            _rows.append((n, LABEL_N[n], *np.log10(band[s:s + WINDOW_CHUNKS].mean(0) + 1e-12)))
BW = pd.DataFrame(_rows, columns=["mission", "class"] + [f"ch{c + 1}" for c in range(N_CH)])
print(f"{len(BW)} steady-flight windows ({WINDOW_CHUNKS} chunks, stride {STRIDE_CHUNKS}) from {BW.mission.nunique()} missions")
_dec = []
for c in range(N_CH):
    col = f"ch{c + 1}"
    x = BW[col].values
    g = x.mean()
    cm = BW.groupby("class")[col].transform("mean").values
    mm = BW.groupby("mission")[col].transform("mean").values
    tot = ((x - g) ** 2).sum()
    _dec.append({"channel": col, "class": ((cm - g) ** 2).sum() / tot, "mission within class": ((mm - cm) ** 2).sum() / tot,
                 "residual (within mission)": ((x - mm) ** 2).sum() / tot})
print("Share of window-level log band-power variance (bins 1-16) explained by:")
print(pd.DataFrame(_dec).set_index("channel").round(3).to_string())

## 4. Representation and windows (fixed a priori; no data-dependent selection)
- **Chunk feature:** `sqrt(|rFFT|) = pow ** 0.25` for bins 1–16 of each of the 5 de-interleaved, mean-removed channels
  → **80 features per chunk**, channel-major. No per-chunk normalization, so loudness is kept. This is v28's A1 feature.
- **Windows:** `WINDOW_CHUNKS = 128` chunks, stride 64. They are defined purely by chunk counts inside one mission,
  with no IMU and no time axis. A window is roughly **2 s *if* a buffer is 16 ms (unverified)**.
- **Tokens:** the mean of each pair of consecutive chunks → **(64, 80)** per window.
- Each window records its mission, class, day, start chunk, the relative position of its centre, and its phase: start
  (< 0.10), steady (0.10–0.90) or end (> 0.90).
- **z-scoring** (per feature) uses the fold's **training windows only**, and is applied to validation and test
  (section 5).

*Cannot show:* whether 16 bins is the best choice. It was deliberately not tuned.

In [ ]:
# ==============================================================================
# 4. WINDOWS AND TOKENS (HOOK 4)
# ==============================================================================
_X, _wm = [], []
for n in NAMES:
    Fm = MISS[n]["feats"]
    N = len(Fm)
    for i, s in enumerate(range(0, N - WINDOW_CHUNKS + 1, STRIDE_CHUNKS)):
        _X.append(Fm[s:s + WINDOW_CHUNKS].reshape(TA, TOKEN_POOL, N_FEAT).mean(axis=1))
        rel = (s + WINDOW_CHUNKS / 2) / N
        _wm.append({"window_id": f"{n}#{i:04d}", "mission": n, "cls": LABEL_N[n], "day": DAY_N[n], "start": s,
                    "end": s + WINDOW_CHUNKS, "mission_chunks": N, "rel_centre": rel,
                    "phase": "start" if rel < STEADY[0] else ("end" if rel > STEADY[1] else "steady")})
X_ALL = np.stack(_X).astype(np.float32)
WMETA = pd.DataFrame(_wm)
del _X, _wm
WIN_ID2ROW = {w: i for i, w in enumerate(WMETA.window_id)}

_len_ok = bool(((WMETA.end - WMETA.start) == WINDOW_CHUNKS).all())
_inside = bool(((WMETA.start >= 0) & (WMETA.end <= WMETA.mission_chunks)).all())
_one_mission = WMETA.groupby("window_id").mission.nunique().max() == 1 and WMETA.window_id.is_unique
_shape_ok = X_ALL.shape[1:] == (TA, N_FEAT)
_finite = bool(np.isfinite(X_ALL).all())
print(f"Tokens: X_ALL {X_ALL.shape} (windows, tokens, features) | NaN/Inf present: {not _finite}")
print(f"Window = {WINDOW_CHUNKS} chunks ≈ {WINDOW_CHUNKS * FRAME / ASSUMED_CHANNEL_RATE_HZ:.2f} s ONLY IF a buffer is "
      f"{1000 * FRAME / ASSUMED_CHANNEL_RATE_HZ:.0f} ms (unverified)")
print("\nWindows per class x phase:")
print(pd.crosstab(WMETA.cls, WMETA.phase).reindex(columns=["start", "steady", "end"]).assign(total=lambda t: t.sum(1)).to_string())
print(f"Windows per mission: median {WMETA.groupby('mission').size().median():.0f}, "
      f"min {WMETA.groupby('mission').size().min()}, max {WMETA.groupby('mission').size().max()}")
_fr = X_ALL.reshape(-1, N_CH, len(KEEP_BINS))
print("\nFeature ranges (sqrt|rFFT|, before z-scoring) per channel: " + " | ".join(
    f"ch{c + 1} min {_fr[:, c].min():.2f} median {np.median(_fr[::7, c]):.2f} max {_fr[:, c].max():.2f}" for c in range(N_CH)))
del _fr
hook(4, _len_ok and _inside and _one_mission and _shape_ok and _finite and len(WMETA) == len(X_ALL),
     f"{len(WMETA)} windows: every window has {WINDOW_CHUNKS} chunks ({_len_ok}), lies inside one mission ({_inside and _one_mission}), "
     f"tokens {X_ALL.shape[1:]} ({_shape_ok}), finite ({_finite})")

## 5. Splits (two protocols, both mission-level)
- **STANDARD:** the identical mission → train/val/test assignment of **v24**. Its `split_missions` function is read
  from `minor-project-v24.ipynb`, checked against its source fingerprint, and executed unchanged (seed 42, 70/15/15 per
  class over all 99 folders). The 4 excluded missions are then dropped, exactly as v24 did. If the file cannot be read,
  **only the day-out protocol runs**, and a loud warning says so.
- **DAY-OUT:** 5 folds. Fold k tests on day k (12-29 is merged into 12-31) and trains on the other four days. Inside
  each fold, ~15 % of the training missions per class are held out for validation. This is **v27's** `_split_inner_val`,
  read from `minor-project-v27.ipynb` and run with day-out protocol index 0. If that file is unreadable, the same rule
  runs with fixed seed 2026. Nothing is selected on the test fold.
- **On Kaggle, attach the folders `version 24/`, `version 27/` and `version 28/` as a dataset.** They are searched under
  `/kaggle/input/**`; locally, under `../version NN/`.
- z-score statistics and class weights are computed here, **from each fold's training windows only**, and a hash is
  printed per fold.

In [ ]:
# ==============================================================================
# 5. SPLITS — STANDARD (v24's assignment) and DAY-OUT (v27's inner-validation rule) (HOOK 3)
# ==============================================================================
def loud(msg):
    print("🚨" * 50)
    for line in msg.split("\n"):
        print("🚨  " + line)
    print("🚨" * 50)


def find_ref_notebook(fname, version):
    cands = sorted(glob.glob(f"/kaggle/input/**/{fname}", recursive=True))
    for base in (Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent):
        cands += [str(base / f"version {version}" / fname), str(base / f"Version {version}" / fname), str(base / fname)]
    cands += [f"/content/{fname}", f"/content/drive/MyDrive/{fname}"]
    return next((Path(c) for c in cands if Path(c).is_file()), None)


def extract_function(nb_path, fname):
    """Source of top-level function `fname` from a notebook's code cells (shell/magic lines are skipped)."""
    try:
        nb = json.loads(Path(nb_path).read_text(encoding="utf-8"))
    except Exception as exc:
        print(f"  could not read {nb_path}: {exc!r}")
        return None
    for cell in nb.get("cells", []):
        if cell.get("cell_type") != "code":
            continue
        src = cell["source"] if isinstance(cell["source"], str) else "".join(cell["source"])
        src = "\n".join(l for l in src.split("\n") if not l.lstrip().startswith(("!", "%")))
        try:
            tree = ast.parse(src)
        except SyntaxError:
            continue
        for node in tree.body:
            if isinstance(node, ast.FunctionDef) and node.name == fname:
                return ast.get_source_segment(src, node)
    return None


def src_sha(src):
    return hashlib.sha1(src.encode()).hexdigest()[:16]


# ---------------- STANDARD: v24's split_missions ----------------
STANDARD_AVAILABLE, STANDARD = False, None
_p24 = find_ref_notebook("minor-project-v24.ipynb", 24)
_src24 = extract_function(_p24, "split_missions") if _p24 else None
if _src24 is None:
    loud("version 24/minor-project-v24.ipynb could NOT be read (not attached?).\n"
         "The STANDARD protocol is SKIPPED. Only the DAY-OUT protocol will run.")
elif src_sha(_src24) != V24_SPLIT_SHA1:
    loud(f"v24 split_missions found at {_p24} but its fingerprint {src_sha(_src24)} != {V24_SPLIT_SHA1}.\n"
         "It is not v24's code, so the STANDARD protocol is SKIPPED. Only DAY-OUT will run.")
else:
    print(f"v24 split_missions read from {_p24} (fingerprint {src_sha(_src24)} == v24) — executing it unchanged:")
    _ns = {"defaultdict": defaultdict, "random": random, "Path": Path, "SPLIT_SEED": SPLIT_SEED,
           "TRAIN_FRACTION": TRAIN_FRACTION, "VAL_FRACTION": VAL_FRACTION}
    exec(_src24, _ns)
    _ms = _ns["split_missions"](DATA_ROOT)
    STANDARD = {s: sorted(m.name for lab in sorted(_ms[s]) for m in _ms[s][lab] if m.name in NAME2DIR)
                for s in ("train", "val", "test")}
    _cnt = {s: len(v) for s, v in STANDARD.items()}
    if _cnt == V24_CORRECTED_SPLIT_COUNTS:
        STANDARD_AVAILABLE = True
        print(f"STANDARD after dropping the 4 excluded missions: {_cnt} == v24's {V24_CORRECTED_SPLIT_COUNTS}")
        print("STANDARD test missions: " + ", ".join(f"{n} (c{LABEL_N[n]})" for n in STANDARD["test"]))
    else:
        loud(f"STANDARD split counts {_cnt} != v24's {V24_CORRECTED_SPLIT_COUNTS}: the dataset listing differs from v24's.\n"
             "The STANDARD protocol is SKIPPED. Only DAY-OUT will run.")
set_all_seeds(SEEDS[0])                         # v24's function re-seeds the global `random`; restore a known state

# ---------------- DAY-OUT: v27's _split_inner_val ----------------
_p27 = find_ref_notebook("minor-project-v27.ipynb", 27)
_src27 = extract_function(_p27, "_split_inner_val") if _p27 else None
if _src27 is not None and src_sha(_src27) == V27_INNER_VAL_SHA1:
    _ns27 = {"np": np, "N_CLASSES": N_CLASSES, "LABEL": LABEL, "INNER_VAL_SEED": INNER_VAL_SEED,
             "INNER_VAL_FRACTION": INNER_VAL_FRACTION}
    exec(_src27, _ns27)
    inner_val = lambda pool, k: _ns27["_split_inner_val"](pool, 0, k)       # protocol index 0 = day_out in v27
    INNER_VAL_RULE = f"v27 _split_inner_val (read from {_p27}, fingerprint OK, seed [{INNER_VAL_SEED}, 0, fold])"
else:
    loud(("v27 notebook not found" if _p27 is None else f"v27 _split_inner_val unreadable or fingerprint mismatch at {_p27}")
         + f"\nFALLBACK inner-validation rule: same per-class ~15% (min 1) mission hold-out with fixed seed "
         f"{INNER_VAL_FALLBACK_SEED}.")

    def inner_val(pool, k):
        rng = np.random.default_rng([INNER_VAL_FALLBACK_SEED, k])
        train, val = [], []
        for c in range(N_CLASSES):
            ms = sorted((m for m in pool if LABEL[m] == c), key=lambda m: m.name)
            perm = rng.permutation(len(ms))
            n_val = max(1, int(round(INNER_VAL_FRACTION * len(ms))))
            val += [ms[i] for i in perm[:n_val]]
            train += [ms[i] for i in perm[n_val:]]
        return train, val

    INNER_VAL_RULE = f"FALLBACK per-class hold-out, seed {INNER_VAL_FALLBACK_SEED}"
print(f"Inner-validation rule: {INNER_VAL_RULE}")

FOLDS = {}
if STANDARD_AVAILABLE:
    FOLDS[("standard", 0)] = {**STANDARD, "fold_day": "v24 standard split"}
for k, day in enumerate(DAYS):
    test = [d for d in KEPT if DAY[d] == day]
    tr, va = inner_val([d for d in KEPT if DAY[d] != day], k)
    FOLDS[("day_out", k)] = {"train": sorted(d.name for d in tr), "val": sorted(d.name for d in va),
                             "test": sorted(d.name for d in test), "fold_day": day}

# ---- leakage asserts (HOOK 3, part 1) ----
_leak_ok, _day_ok = True, True
for key, f in FOLDS.items():
    tr, va, te = set(f["train"]), set(f["val"]), set(f["test"])
    _leak_ok &= not (tr & va) and not (tr & te) and not (va & te) and (tr | va | te) == set(NAMES)
    if key[0] == "day_out":
        _day_ok &= not ({DAY_N[n] for n in tr | va} & {DAY_N[n] for n in te}) and {DAY_N[n] for n in te} == {f["fold_day"]}
_partition = sorted(n for k in range(len(DAYS)) for n in FOLDS[("day_out", k)]["test"]) == sorted(NAMES)
hook(3, _leak_ok, "no mission is in more than one of train/val/test in any fold; each fold covers all 95 missions")
hook(3, _day_ok and _partition, "day-out: the test day never appears in train/val; the 5 test folds partition the 95 missions")

# ---- z-score statistics + class weights from TRAINING windows only (HOOK 3, part 2) ----
FOLD_PREP = {}
_z_ok = True
for key, f in FOLDS.items():
    idx = {s: np.where(WMETA.mission.isin(f[s]).values)[0] for s in ("train", "val", "test")}
    used = set(WMETA.mission.values[idx["train"]])
    _z_ok &= used == set(f["train"]) and not (used & (set(f["val"]) | set(f["test"])))
    Xtr = X_ALL[idx["train"]].reshape(-1, N_FEAT)
    mu = Xtr.mean(0, dtype=np.float64).astype(np.float32)
    sd = (Xtr.std(0, dtype=np.float64) + 1e-6).astype(np.float32)
    del Xtr
    counts = np.bincount(WMETA.cls.values[idx["train"]], minlength=N_CLASSES)
    cw = (counts.sum() / (N_CLASSES * np.maximum(counts, 1))).astype(np.float32)      # v28: inverse class frequency
    FOLD_PREP[key] = {"idx": idx, "mu": mu, "sd": sd, "zhash": sha16(mu, sd), "class_weights": cw, "train_counts": counts,
                      "train_missions_hash": hashlib.sha1("\n".join(sorted(used)).encode()).hexdigest()[:12]}

print("\n" + "=" * 150)
print(f"{'fold':<16}{'test':<20}{'missions tr/val/test':<22}{'test missions per class':<26}{'windows tr/val/test':<22}"
      f"{'z-score hash':<18}{'class weights (from training windows)'}")
print("=" * 150)
for key, f in FOLDS.items():
    P = FOLD_PREP[key]
    per_class = np.bincount([LABEL_N[n] for n in f["test"]], minlength=N_CLASSES).tolist()
    print(f"{key[0] + ' ' + str(key[1]):<16}{f['fold_day']:<20}"
          f"{'/'.join(str(len(f[s])) for s in ('train', 'val', 'test')):<22}{str(per_class):<26}"
          f"{'/'.join(str(len(P['idx'][s])) for s in ('train', 'val', 'test')):<22}{P['zhash']:<18}"
          f"{np.round(P['class_weights'], 3).tolist()}" + ("   <- SMALL fold (10 missions)" if f["fold_day"] == SMALL_FOLD_DAY else ""))
    print(f"{'':<16}train missions per class {np.bincount([LABEL_N[n] for n in f['train']], minlength=N_CLASSES).tolist()}, "
          f"val missions per class {np.bincount([LABEL_N[n] for n in f['val']], minlength=N_CLASSES).tolist()}")
hook(3, _z_ok, "z-score mean/std computed from each fold's TRAINING windows only (hashes above); applied unchanged to val/test")

## 6. Model and training
- **Real `mamba_ssm.Mamba` only**, asserted.
- **Architecture:** v28's `UnimodalMamba` audio path: encoder `Linear(80, 256) → LayerNorm → GELU → Linear(256, 256)`,
  a modality embedding + LayerNorm, then **2 pre-norm residual Mamba blocks** (d_state 16, d_conv 2, expand 2). After
  that come attention pooling `Linear(256, 64) → Tanh → Linear(64, 1)` and v28's classifier head
  `Dropout(0.15) → Linear(256, 128) → GELU → Linear(128, 5)`.
- **Training:** exactly v28's rules. AdamW (lr 1e-4, weight decay 1e-4); cosine annealing to 1e-6 over 20 epochs,
  stepped per epoch; grad clip 1.0; batch 32; **20 epochs, no early stopping**. The checkpoint kept is the epoch with
  the best validation macro-F1.
- **Loss:** class-weighted cross-entropy, with weights from the fold's training windows (printed in section 5).
- Each run builds a fresh model, prints an initial-weights fingerprint, and frees GPU memory afterwards.
- Every run is appended to `audio_only_results.csv` as soon as it finishes. The loop is resumable: rows already in the
  CSV are skipped.

In [ ]:
# ==============================================================================
# 6a. MODEL (v28 UnimodalMamba, audio path) + v28 settings cross-check (HOOK 6)
# ==============================================================================
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, confusion_matrix, f1_score,
                             precision_recall_fscore_support, roc_auc_score)

device = torch.device("cuda")

# ---- cross-check the transcribed settings against version 28's config cell, if the notebook is attached ----
V28_KEYS = ["D_MODEL", "DROPOUT", "MAMBA_NUM_BLOCKS", "MAMBA_D_STATE", "MAMBA_D_CONV", "MAMBA_EXPAND", "EXPECTED_MAMBA_PARAMS",
            "EPOCHS", "BATCH_SIZE", "LEARNING_RATE", "WEIGHT_DECAY", "LR_MIN", "GRAD_CLIP_NORM", "NUM_WORKERS"]
_p28 = find_ref_notebook("minor-project-v28.ipynb", 28)
if _p28 is not None:
    _v28 = {}
    for cell in json.loads(_p28.read_text(encoding="utf-8")).get("cells", []):
        if cell.get("cell_type") != "code":
            continue
        src = "".join(cell["source"]) if isinstance(cell["source"], list) else cell["source"]
        try:
            tree = ast.parse("\n".join(l for l in src.split("\n") if not l.lstrip().startswith(("!", "%"))))
        except SyntaxError:
            continue
        for node in tree.body:
            if isinstance(node, ast.Assign) and len(node.targets) == 1 and isinstance(node.targets[0], ast.Name) \
                    and node.targets[0].id in V28_KEYS and node.targets[0].id not in _v28:
                try:
                    _v28[node.targets[0].id] = ast.literal_eval(node.value)
                except ValueError:
                    pass
    _diff = {k: (globals()[k], _v28.get(k)) for k in V28_KEYS if _v28.get(k) != globals()[k]}
    hook(6, not _diff, f"training settings == version 28's config cell ({_p28})" + (f" — DIFFERENCES {_diff}" if _diff else ""),
         stop=False)
else:
    print("version 28 notebook not attached: using the settings transcribed from version 28/minor-project-v28.ipynb:")
print("  " + ", ".join(f"{k}={globals()[k]}" for k in V28_KEYS))


class MambaBlock(nn.Module):
    """Pre-norm residual wrapper around the real mamba_ssm.Mamba selective-scan SSM (v28)."""

    def __init__(self):
        super().__init__()
        self.norm = nn.LayerNorm(D_MODEL)
        self.mamba = MAMBA_CLASS(d_model=D_MODEL, d_state=MAMBA_D_STATE, d_conv=MAMBA_D_CONV, expand=MAMBA_EXPAND)

    def forward(self, x):
        return x + self.mamba(self.norm(x))


class AudioMamba(nn.Module):
    """v28 UnimodalMamba, audio path: encoder -> + modality embedding -> LayerNorm -> 2 Mamba blocks -> attention
    pooling -> classifier."""

    def __init__(self, in_dim=N_FEAT):
        super().__init__()
        self.audio_encoder = nn.Sequential(nn.Linear(in_dim, D_MODEL), nn.LayerNorm(D_MODEL), nn.GELU(), nn.Linear(D_MODEL, D_MODEL))
        self.modality_embedding = nn.Embedding(1, D_MODEL)
        self.token_norm = nn.LayerNorm(D_MODEL)
        self.mamba_blocks = nn.ModuleList([MambaBlock() for _ in range(MAMBA_NUM_BLOCKS)])
        self.pool_attn = nn.Sequential(nn.Linear(D_MODEL, 64), nn.Tanh(), nn.Linear(64, 1))
        self.classifier = nn.Sequential(nn.Dropout(DROPOUT), nn.Linear(D_MODEL, D_MODEL // 2), nn.GELU(),
                                        nn.Linear(D_MODEL // 2, N_CLASSES))

    def forward(self, x):
        h = self.token_norm(self.audio_encoder(x) + self.modality_embedding.weight.view(1, 1, -1))
        for block in self.mamba_blocks:
            h = block(h)
        w = F.softmax(self.pool_attn(h), dim=1)
        return self.classifier((h * w).sum(dim=1))


def verify_model(model):
    real = all(type(b.mamba) is MAMBA_CLASS for b in model.mamba_blocks) and MAMBA_CLASS.__module__.startswith("mamba_ssm")
    blocks_ok = len(model.mamba_blocks) == MAMBA_NUM_BLOCKS and all(
        (b.mamba.d_state, b.mamba.d_conv, b.mamba.expand) == (MAMBA_D_STATE, MAMBA_D_CONV, MAMBA_EXPAND) for b in model.mamba_blocks)
    n_mamba = sum(p.numel() for p in model.mamba_blocks.parameters())
    n_params = sum(p.numel() for p in model.parameters())
    return real and blocks_ok and n_mamba == EXPECTED_MAMBA_PARAMS, n_params, n_mamba


set_all_seeds(SEEDS[0])
_probe = AudioMamba().to(device)
_ok, N_PARAMS, _n_mamba = verify_model(_probe)
assert _ok, "model is not the expected real-mamba_ssm architecture"
print(f"AudioMamba: {N_PARAMS:,} parameters (Mamba blocks {_n_mamba:,} == {EXPECTED_MAMBA_PARAMS:,}) | block class "
      f"{MAMBA_CLASS.__module__}.{MAMBA_CLASS.__name__} | modules {[n for n, _ in _probe.named_children()]}")
with torch.no_grad():
    assert _probe(torch.zeros(2, TA, N_FEAT, device=device)).shape == (2, N_CLASSES)
hook(6, _ok, f"every block is mamba_ssm.Mamba (d_state {MAMBA_D_STATE}, d_conv {MAMBA_D_CONV}); {N_PARAMS:,} parameters",
     stop=False)
del _probe
torch.cuda.empty_cache()

In [ ]:
# ==============================================================================
# 6b. TRAINING FUNCTION + optional determinism check (HOOK 6)
# ==============================================================================
def make_loader(X, y, shuffle, seed):
    ds = TensorDataset(torch.from_numpy(X), torch.from_numpy(y))
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle, num_workers=NUM_WORKERS, pin_memory=True,
                      worker_init_fn=worker_init_fn, generator=make_generator(seed) if shuffle else None)


@torch.no_grad()
def predict(model, loader):
    model.eval()
    out = [F.softmax(model(x.to(device, non_blocking=True)), dim=-1).float().cpu().numpy() for x, _ in loader]
    return np.concatenate(out)


def compute_metrics(y, P):
    pred = P.argmax(1)
    try:
        auroc = float(roc_auc_score(y, P, multi_class="ovr", average="macro", labels=list(range(N_CLASSES))))
    except ValueError:
        auroc = float("nan")
    return {"accuracy": float(accuracy_score(y, pred)),
            "macro_f1": float(f1_score(y, pred, average="macro", labels=list(range(N_CLASSES)), zero_division=0)),
            "balanced_accuracy": float(balanced_accuracy_score(y, pred)), "macro_auroc": auroc}


def mission_level(missions, y, P):
    """Mission prediction = argmax of the mean window probabilities."""
    df = pd.DataFrame(P, columns=range(N_CLASSES))
    df["mission"], df["y"] = missions, y
    g = df.groupby("mission", sort=True)
    return g.y.first().values, g[list(range(N_CLASSES))].mean().values, g.y.first().index.values


def train_run(key, seed, epochs=EPOCHS, save=True, verbose=True):
    t0 = time.time()
    prep, fold = FOLD_PREP[key], FOLDS[key]
    idx = prep["idx"]
    Xs = {s: ((X_ALL[idx[s]] - prep["mu"]) / prep["sd"]).astype(np.float32) for s in ("train", "val", "test")}
    ys = {s: WMETA.cls.values[idx[s]].astype(np.int64) for s in ("train", "val", "test")}
    set_all_seeds(seed)
    model = AudioMamba().to(device)
    init_sha1 = hashlib.sha1(b"".join(v.detach().cpu().contiguous().numpy().tobytes()
                                      for v in model.state_dict().values())).hexdigest()[:12]
    ok, n_params, _ = verify_model(model)
    assert ok, "not the expected real-mamba_ssm model"
    criterion = nn.CrossEntropyLoss(weight=torch.tensor(prep["class_weights"], device=device))
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=LR_MIN)
    tr = make_loader(Xs["train"], ys["train"], True, seed)
    va = make_loader(Xs["val"], ys["val"], False, seed)
    te = make_loader(Xs["test"], ys["test"], False, seed)
    tag = f"[{key[0]} {key[1]} | seed {seed}]"
    if verbose:
        print(f"{tag} init weights {init_sha1} | {n_params:,} params | windows train {len(ys['train'])} / val "
              f"{len(ys['val'])} / test {len(ys['test'])} | z-score {prep['zhash']}")
    best_f1, best_state, best_epoch, history = -1.0, None, None, []
    for epoch in range(1, epochs + 1):
        model.train()
        running, nb = 0.0, 0
        for x, y in tr:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            optimizer.zero_grad()
            loss = criterion(model(x), y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=GRAD_CLIP_NORM)
            optimizer.step()
            running += loss.item()
            nb += 1
        scheduler.step()
        assert math.isfinite(running), f"{tag} non-finite training loss"
        val_f1 = compute_metrics(ys["val"], predict(model, va))["macro_f1"]
        history.append((running / nb, val_f1))
        improved = val_f1 > best_f1
        if improved:
            best_f1, best_state, best_epoch = val_f1, copy.deepcopy(model.state_dict()), epoch
        if verbose:
            print(f"{tag} epoch {epoch:02d}/{epochs} | train CE {running / nb:.4f} | val macro-F1 {val_f1:.4f}"
                  + ("  * best" if improved else ""))
    model.load_state_dict(best_state)
    P = predict(model, te)
    wm = compute_metrics(ys["test"], P)
    my, mP, _ = mission_level(WMETA.mission.values[idx["test"]], ys["test"], P)
    mm = compute_metrics(my, mP)
    name = f"{key[0]}_{key[1]}_{seed}"
    res = {"protocol": key[0], "fold": key[1], "fold_day": fold["fold_day"], "seed": seed,
           **{f"n_{s}_missions": len(fold[s]) for s in ("train", "val", "test")},
           **{f"n_{s}_windows": len(ys[s]) for s in ("train", "val", "test")},
           **wm, **{f"mission_{k}": v for k, v in mm.items()},
           "best_epoch": best_epoch, "best_val_macro_f1": round(best_f1, 6), "init_sha1": init_sha1,
           "zstats_sha1": prep["zhash"], "class_weights": json.dumps(np.round(prep["class_weights"], 4).tolist()),
           "n_params": n_params, "train_seconds": round(time.time() - t0, 1), "history": history}
    if save:
        m = WMETA.iloc[idx["test"]]
        np.savez(PROBS_DIR / f"{name}.npz", window_id=m.window_id.values.astype(str), mission=m.mission.values.astype(str),
                 cls=m.cls.values, day=m.day.values.astype(str), phase=m.phase.values.astype(str), start=m.start.values,
                 probs=P.astype(np.float32))
        torch.save(best_state, CKPT_DIR / f"{name}.pt")
        res["probs_file"] = str(PROBS_DIR / f"{name}.npz")
        res["checkpoint"] = str(CKPT_DIR / f"{name}.pt")
    del model, optimizer, scheduler, best_state, tr, va, te, Xs
    gc.collect()
    torch.cuda.empty_cache()
    return res


DETERMINISM = None
if CHECK_DETERMINISM:
    _key = ("standard", 0) if STANDARD_AVAILABLE else ("day_out", 1)
    print(f"Determinism check: {_key}, seed 42, {DETERMINISM_EPOCHS} epochs, twice ...")
    _r1 = train_run(_key, 42, epochs=DETERMINISM_EPOCHS, save=False, verbose=False)
    _r2 = train_run(_key, 42, epochs=DETERMINISM_EPOCHS, save=False, verbose=False)
    DETERMINISM = _r1["history"] == _r2["history"] and _r1["init_sha1"] == _r2["init_sha1"]
    for e, (a, b) in enumerate(zip(_r1["history"], _r2["history"]), 1):
        print(f"  epoch {e}: run1 CE {a[0]:.8f} val-F1 {a[1]:.6f} | run2 CE {b[0]:.8f} val-F1 {b[1]:.6f}")
    hook(6, DETERMINISM, f"determinism: identical initial weights ({_r1['init_sha1']}) and bit-identical per-epoch losses "
                         f"over two {DETERMINISM_EPOCHS}-epoch runs" + ("" if DETERMINISM else
                         " — NOT identical: mamba-ssm's CUDA backward may use non-deterministic atomics; results are "
                         "then reproducible only up to that noise"), stop=False)
else:
    print("CHECK_DETERMINISM = False: determinism check skipped.")

In [ ]:
# ==============================================================================
# 6c. RUNS — STANDARD x 3 seeds, DAY-OUT x 5 folds x 3 seeds; CSV append per run; resumable
# ==============================================================================
CSV_FIELDS = ["protocol", "fold", "fold_day", "seed", "n_train_missions", "n_val_missions", "n_test_missions",
              "n_train_windows", "n_val_windows", "n_test_windows", "accuracy", "macro_f1", "balanced_accuracy", "macro_auroc",
              "mission_accuracy", "mission_macro_f1", "mission_balanced_accuracy", "mission_macro_auroc", "best_epoch",
              "best_val_macro_f1", "init_sha1", "zstats_sha1", "class_weights", "n_params", "train_seconds", "probs_file",
              "checkpoint", "finished_at"]


def load_done():
    if not RESULTS_CSV.exists():
        return {}
    df = pd.read_csv(RESULTS_CSV)
    return {(r.protocol, int(r.fold), int(r.seed)): r for r in df.itertuples()
            if (PROBS_DIR / f"{r.protocol}_{int(r.fold)}_{int(r.seed)}.npz").exists()}


def append_row(res):
    new = not RESULTS_CSV.exists()
    with open(RESULTS_CSV, "a", newline="") as f:
        w = csv.DictWriter(f, fieldnames=CSV_FIELDS, extrasaction="ignore")
        if new:
            w.writeheader()
        w.writerow({**res, "finished_at": datetime.now().isoformat(timespec="seconds")})


RUNS = ([("standard", 0, s) for s in SEEDS] if STANDARD_AVAILABLE else []) + \
       [("day_out", k, s) for s in SEEDS for k in range(len(DAYS))]          # seed-major: complete fold sets per seed
_done = load_done()
_todo = [r for r in RUNS if r not in _done]
print(f"{len(RUNS)} runs planned ({'3 standard + ' if STANDARD_AVAILABLE else 'NO standard (v24 unreadable) + '}"
      f"{len(DAYS) * len(SEEDS)} day-out) | already in CSV: {len(RUNS) - len(_todo)} | to train now: {len(_todo)}")
_t_start, _n_trained = time.time(), 0
for i, (p, k, s) in enumerate(RUNS, 1):
    if (p, k, s) in _done:
        r = _done[(p, k, s)]
        print(f"[run {i:02d}/{len(RUNS)}] {p} fold {k} ({r.fold_day}) seed {s} — already in CSV (macro-F1 {r.macro_f1:.4f}), skipped")
        continue
    res = train_run((p, k), s)
    append_row(res)
    _n_trained += 1
    _el = time.time() - _t_start
    _left = len(_todo) - _n_trained
    print(f"[run {i:02d}/{len(RUNS)}] {p} fold {k} ({res['fold_day']}) seed {s} | window macro-F1 {res['macro_f1']:.4f} | "
          f"mission macro-F1 {res['mission_macro_f1']:.4f} | best epoch {res['best_epoch']} | {res['train_seconds']:.0f} s | "
          f"ETA {_left * _el / _n_trained / 60:.0f} min")
print(f"All runs done: {len(load_done())}/{len(RUNS)} in the CSV")

## 7. Results
Everything is read back from the per-run probability files, so this section also works after a resumed session.
- **7a — STANDARD split.** Per seed and mean ± SD, at window and mission level. Mission prediction = argmax of the mean
  window probabilities.
- **7b — DAY-OUT, pooled out-of-fold.** Each window's probabilities are averaged over the seeds. CIs come from a
  **mission-level cluster bootstrap**: 2000 resamples of missions within class, never windows. Per-fold (per-day)
  macro-F1 and balanced accuracy are reported. Per-fold per-class recall is **not** interpreted: each day has only 2–8
  missions per class.
- **7c** — safety-relevant errors. **7d** — where audio fails: by phase, by day, and the top confusion pairs.
- **7e** — evidence accumulation over a mission's first k windows. **7f** — context from v28 (not a comparison
  experiment). **7g** — a summary generated from the numbers.

In [ ]:
# ==============================================================================
# 7. HELPERS + 7a STANDARD SPLIT
# ==============================================================================
def load_run(p, k, s):
    z = np.load(PROBS_DIR / f"{p}_{k}_{s}.npz")
    return {key: z[key] for key in z.files}


def prf_from_cm(cm):
    tp = np.diag(cm).astype(float)
    prec = np.divide(tp, cm.sum(0), out=np.zeros_like(tp), where=cm.sum(0) > 0)
    rec = np.divide(tp, cm.sum(1), out=np.zeros_like(tp), where=cm.sum(1) > 0)
    f1 = np.divide(2 * prec * rec, prec + rec, out=np.zeros_like(tp), where=(prec + rec) > 0)
    return prec, rec, f1


def per_class_table(y, pred):
    p, r, f, n = precision_recall_fscore_support(y, pred, labels=list(range(N_CLASSES)), zero_division=0)
    return pd.DataFrame({"precision": p, "recall": r, "f1": f, "support": n}, index=[f"class {c}" for c in range(N_CLASSES)])


def plot_cm(ax, cm, title):
    cmn = cm / np.maximum(cm.sum(1, keepdims=True), 1)
    ax.imshow(cmn, cmap=SEQ_CMAP, vmin=0, vmax=1)
    ax.grid(False)
    for i in range(N_CLASSES):
        for j in range(N_CLASSES):
            ax.text(j, i, f"{int(cm[i, j])}\n{cmn[i, j]:.2f}", ha="center", va="center", fontsize=7,
                    color="white" if cmn[i, j] > 0.6 else "#0b0b0b")
    ax.set_xticks(range(N_CLASSES))
    ax.set_yticks(range(N_CLASSES))
    ax.set_xlabel("predicted class")
    ax.set_ylabel("true class")
    ax.set_title(title, fontsize=9)


def fmt_ms(v):
    v = np.asarray(v, dtype=float)
    return f"{np.nanmean(v):.4f} ± {np.nanstd(v, ddof=1) if len(v) > 1 else 0:.4f}"


_done = load_done()
STD_RESULTS = None
if STANDARD_AVAILABLE and all(("standard", 0, s) in _done for s in SEEDS):
    _rows, _pcw, _pcm, _cmw, _cmm = [], [], [], np.zeros((N_CLASSES, N_CLASSES), int), np.zeros((N_CLASSES, N_CLASSES), int)
    for s in SEEDS:
        r = load_run("standard", 0, s)
        y, P = r["cls"], r["probs"]
        my, mP, _ = mission_level(r["mission"], y, P)
        _rows.append({"seed": s, **{f"window {k}": v for k, v in compute_metrics(y, P).items()},
                      **{f"mission {k}": v for k, v in compute_metrics(my, mP).items()}})
        _pcw.append(per_class_table(y, P.argmax(1)))
        _pcm.append(per_class_table(my, mP.argmax(1)))
        _cmw += confusion_matrix(y, P.argmax(1), labels=list(range(N_CLASSES)))
        _cmm += confusion_matrix(my, mP.argmax(1), labels=list(range(N_CLASSES)))
    STD_RESULTS = pd.DataFrame(_rows).set_index("seed")
    print("=== 7a. STANDARD split (v24's mission assignment; 15 test missions) — per seed ===")
    print(STD_RESULTS.round(4).to_string())
    print("mean ± SD over seeds: " + " | ".join(f"{c} {fmt_ms(STD_RESULTS[c])}" for c in STD_RESULTS.columns))
    for lvl, tabs in (("WINDOW", _pcw), ("MISSION", _pcm)):
        m = sum(t[["precision", "recall", "f1"]] for t in tabs) / len(tabs)
        sd = pd.concat([t[["precision", "recall", "f1"]] for t in tabs]).groupby(level=0).std(ddof=1)
        print(f"\nPer-class precision / recall / F1 at {lvl} level, mean (SD) over {len(tabs)} seeds "
              f"(support per seed: {tabs[0]['support'].tolist()}):")
        print((m.round(3).astype(str) + " (" + sd.round(3).astype(str) + ")").to_string())
    fig, axs = plt.subplots(1, 2, figsize=(10, 4.3))
    plot_cm(axs[0], _cmw, f"window level, summed over {len(SEEDS)} seeds")
    plot_cm(axs[1], _cmm, f"mission level, summed over {len(SEEDS)} seeds")
    fig.suptitle("STANDARD split — confusion matrices (counts and row-normalized recall)")
    save_fig(fig, "7a_standard_confusion", "Standard split (v24 assignment), 15 test missions per seed; the three seeds share "
                                           "the same test missions, so the counts are summed over seeds.")
else:
    print("7a skipped: the STANDARD protocol did not run (v24 split unreadable) or its runs are incomplete.")

In [ ]:
# ==============================================================================
# 7b. DAY-OUT — pooled out-of-fold (seed-averaged probabilities), mission-level cluster bootstrap CIs
# ==============================================================================
_seeds_do = [s for s in SEEDS if all(("day_out", k, s) in _done for k in range(len(DAYS)))]
assert _seeds_do, "no complete day-out seed: nothing to analyse"
if len(_seeds_do) < len(SEEDS):
    loud(f"Only seeds {_seeds_do} have all 5 day-out folds; the pooled results below use those seeds only.")
_per_seed = {}
for s in _seeds_do:
    parts = [load_run("day_out", k, s) for k in range(len(DAYS))]
    ids = np.concatenate([p["window_id"] for p in parts])
    assert len(ids) == len(set(ids)) == len(WMETA), f"seed {s}: pooled windows {len(ids)} vs {len(WMETA)}"
    order = np.argsort(ids)
    _per_seed[s] = {"ids": ids[order], "P": np.concatenate([p["probs"] for p in parts])[order]}
POOL_IDS = _per_seed[_seeds_do[0]]["ids"]
assert all(np.array_equal(v["ids"], POOL_IDS) for v in _per_seed.values())
POOL = WMETA.iloc[[WIN_ID2ROW[w] for w in POOL_IDS]].reset_index(drop=True)
POOL_P = np.mean([v["P"] for v in _per_seed.values()], axis=0)
POOL_Y = POOL.cls.values
POOL_PRED = POOL_P.argmax(1)


def cluster_bootstrap(missions, y, pred, B=BOOTSTRAP_RESAMPLES, seed=BOOTSTRAP_SEED):
    """Resample MISSIONS within class (never windows). Returns bootstrap confusion matrices (B, 5, 5)."""
    rng = np.random.default_rng(seed)
    ms = pd.Series(missions).unique()
    cnt = np.zeros((len(ms), N_CLASSES), int)
    pos = {m: i for i, m in enumerate(ms)}
    mcls = np.zeros(len(ms), int)
    for m, t, p in zip(missions, y, pred):
        cnt[pos[m], p] += 1
        mcls[pos[m]] = t
    cms = np.zeros((B, N_CLASSES, N_CLASSES))
    for c in range(N_CLASSES):
        rows = np.where(mcls == c)[0]
        draw = rows[rng.integers(0, len(rows), (B, len(rows)))]
        cms[:, c, :] = cnt[draw].sum(1)
    return cms


def boot_summary(missions, y, pred):
    cm = confusion_matrix(y, pred, labels=list(range(N_CLASSES)))
    p, r, f = prf_from_cm(cm)
    cms = cluster_bootstrap(missions, y, pred)
    bp, br, bf = zip(*(prf_from_cm(c) for c in cms))
    bp, br, bf = np.array(bp), np.array(br), np.array(bf)
    ci = lambda a: np.percentile(a, [2.5, 97.5], axis=0)
    tab = pd.DataFrame({"precision": p, "P CI": [f"[{a:.3f}, {b:.3f}]" for a, b in ci(bp).T],
                        "recall": r, "R CI": [f"[{a:.3f}, {b:.3f}]" for a, b in ci(br).T],
                        "f1": f, "F1 CI": [f"[{a:.3f}, {b:.3f}]" for a, b in ci(bf).T]},
                       index=[f"class {c}" for c in range(N_CLASSES)])
    macro = (f.mean(), *ci(bf.mean(1)))
    bal = (r.mean(), *ci(br.mean(1)))
    return tab, macro, bal, cm, (r, ci(br))


DO_TAB, DO_MACRO, DO_BAL, DO_CM, DO_RECALL = boot_summary(POOL.mission.values, POOL_Y, POOL_PRED)
print(f"=== 7b. DAY-OUT pooled out-of-fold, window level (probabilities averaged over seeds {_seeds_do}; "
      f"{len(POOL)} windows, 95 missions) ===")
print(DO_TAB.round(3).to_string())
print(f"Pooled macro-F1 {DO_MACRO[0]:.4f} [95% CI {DO_MACRO[1]:.4f}, {DO_MACRO[2]:.4f}] | balanced accuracy "
      f"{DO_BAL[0]:.4f} [{DO_BAL[1]:.4f}, {DO_BAL[2]:.4f}] | accuracy {accuracy_score(POOL_Y, POOL_PRED):.4f} | "
      f"macro-AUROC {compute_metrics(POOL_Y, POOL_P)['macro_auroc']:.4f}")

POOL_SEED_F1 = {s: f1_score(POOL_Y, v["P"].argmax(1), average="macro", labels=list(range(N_CLASSES)), zero_division=0)
                for s, v in _per_seed.items()}
print("Per-seed pooled macro-F1: " + ", ".join(f"seed {s} {v:.4f}" for s, v in POOL_SEED_F1.items())
      + f" | seed-to-seed SD {np.std(list(POOL_SEED_F1.values()), ddof=1) if len(POOL_SEED_F1) > 1 else float('nan'):.4f}")

MY, MP, MNAMES = mission_level(POOL.mission.values, POOL_Y, POOL_P)
DO_MTAB, DO_MMACRO, DO_MBAL, DO_MCM, _ = boot_summary(MNAMES, MY, MP.argmax(1))
print(f"\nMission level (argmax of mean window probabilities, 95 missions): macro-F1 {DO_MMACRO[0]:.4f} "
      f"[{DO_MMACRO[1]:.4f}, {DO_MMACRO[2]:.4f}] | balanced accuracy {DO_MBAL[0]:.4f} | accuracy {accuracy_score(MY, MP.argmax(1)):.4f}")
print(DO_MTAB.round(3).to_string())

print("\nPer fold (= per unseen day), seed-averaged probabilities; per-seed mean ± SD in brackets. Per-class recall per "
      "fold is NOT interpreted (2–8 missions per class):")
FOLD_TAB = []
for k, day in enumerate(DAYS):
    sel = (POOL.day == day).values
    m = compute_metrics(POOL_Y[sel], POOL_P[sel])
    per_seed = [compute_metrics(POOL_Y[sel], v["P"][sel]) for v in _per_seed.values()]
    FOLD_TAB.append({"fold": k, "test day": day, "missions": POOL[sel].mission.nunique(), "windows": int(sel.sum()),
                     "macro-F1": m["macro_f1"], "balanced acc": m["balanced_accuracy"], "accuracy": m["accuracy"],
                     "macro-F1 per seed": fmt_ms([q["macro_f1"] for q in per_seed]),
                     "note": "SMALL fold (10 missions)" if day == SMALL_FOLD_DAY else ""})
FOLD_TAB = pd.DataFrame(FOLD_TAB).set_index("fold")
print(FOLD_TAB.round(4).to_string())

In [ ]:
# ==============================================================================
# 7c. SAFETY-RELEVANT ERRORS (day-out, pooled, window level) + pooled confusion matrix
# ==============================================================================
_faulty = POOL_Y > 0
_err = POOL_PRED != POOL_Y
SAFETY = {"faulty windows predicted Normal (missed fault)": float((POOL_PRED[_faulty] == 0).mean()),
          "Normal windows predicted faulty (false alarm)": float((POOL_PRED[~_faulty] > 0).mean()),
          "errors that are adjacent-class (|pred − true| = 1)": float((np.abs(POOL_PRED - POOL_Y)[_err] == 1).mean()) if _err.any() else float("nan"),
          "errors of >= 2 severity levels": float((np.abs(POOL_PRED - POOL_Y)[_err] >= 2).mean()) if _err.any() else float("nan")}
print("=== 7c. Safety-relevant errors, day-out pooled (window level) ===")
for k, v in SAFETY.items():
    print(f"  {k:<55} {v:.3f}")
_mf = MY > 0
print(f"  mission level: faulty missions predicted Normal {(MP.argmax(1)[_mf] == 0).mean():.3f} | "
      f"Normal missions predicted faulty {(MP.argmax(1)[~_mf] > 0).mean():.3f}")
print("\nPooled confusion matrix (rows = true, cols = predicted, windows):")
print(pd.DataFrame(DO_CM, index=[f"true {c}" for c in range(N_CLASSES)], columns=[f"pred {c}" for c in range(N_CLASSES)]).to_string())
fig, axs = plt.subplots(1, 2, figsize=(10, 4.3))
plot_cm(axs[0], DO_CM, "window level")
plot_cm(axs[1], DO_MCM, "mission level")
fig.suptitle("DAY-OUT pooled out-of-fold confusion matrices (seed-averaged probabilities)")
save_fig(fig, "7c_dayout_confusion", "Every mission is tested exactly once, on a model that never saw its recording day. "
                                     "Cell text = count and row-normalized recall.")

In [ ]:
# ==============================================================================
# 7d. WHERE AUDIO FAILS — by phase, by day, top confusion pairs (+ pointer to section 3b)
# ==============================================================================
print("=== 7d. Day-out pooled performance by flight phase (window centre: start < 10%, steady 10–90%, end > 90%) ===")
_ph = []
for ph in ["start", "steady", "end"]:
    sel = (POOL.phase == ph).values
    m = compute_metrics(POOL_Y[sel], POOL_P[sel])
    _ph.append({"phase": ph, "windows": int(sel.sum()), "accuracy": m["accuracy"], "macro-F1": m["macro_f1"],
                "balanced acc": m["balanced_accuracy"], "missed faults": float((POOL_PRED[sel & _faulty] == 0).mean())})
PHASE_TAB = pd.DataFrame(_ph).set_index("phase")
print(PHASE_TAB.round(4).to_string())
print("\nBy unseen day (from 7b):")
print(FOLD_TAB[["test day", "macro-F1", "balanced acc", "accuracy", "note"]].round(4).to_string())

_sym = DO_CM + DO_CM.T
TOP_PAIRS = sorted(((int(_sym[i, j]), i, j) for i in range(N_CLASSES) for j in range(i + 1, N_CLASSES)), reverse=True)[:2]
print("\nTop two confusion pairs (windows confused in either direction):")
for n_conf, i, j in TOP_PAIRS:
    print(f"\n  class {i} <-> class {j}: {n_conf} windows ({DO_CM[i, j]} true {i} -> {j}, {DO_CM[j, i]} true {j} -> {i})")
    if (i, j) in D_TABLES:
        print(f"  -> matching descriptive table: section 3b, 'class {i} v {j}' (repeated here):")
        print(D_TABLES[(i, j)].to_string(index=False))
    else:
        print(f"  -> non-adjacent pair, not among the section-3b tables; the same computation (section 3b's cohen_top_table):")
        print(cohen_top_table(i, j)[1].to_string(index=False))

In [ ]:
# ==============================================================================
# 7e. EVIDENCE ACCUMULATION (day-out): average a mission's first k windows in order
# ==============================================================================
_ev_rows = []
_by_m = {m: g.sort_values("start").index.values for m, g in POOL.groupby("mission")}
_ks = EVIDENCE_K + ["all"]
for k in _ks:
    ys, preds, short = [], [], 0
    for m, rows in _by_m.items():
        use = rows if k == "all" else rows[:k]
        short += int(k != "all" and len(rows) < k)
        ys.append(POOL_Y[rows[0]])
        preds.append(POOL_P[use].mean(0).argmax())
    ys, preds = np.array(ys), np.array(preds)
    _ev_rows.append({"k": k, "mission accuracy": accuracy_score(ys, preds),
                     "mission macro-F1": f1_score(ys, preds, average="macro", labels=list(range(N_CLASSES)), zero_division=0),
                     "missions with < k windows (use all)": short})
EVID = pd.DataFrame(_ev_rows).set_index("k")
print("=== 7e. Evidence accumulation, day-out pooled (seed-averaged), 95 missions ===")
print(EVID.round(4).to_string())
fig, ax = plt.subplots(figsize=(7.5, 4))
_x = list(range(len(_ks)))
ax.plot(_x, EVID["mission macro-F1"].values, marker="o", ms=8, color=CLASS_COLORS[0], label="mission macro-F1")
ax.plot(_x, EVID["mission accuracy"].values, marker="s", ms=8, color=CLASS_COLORS[1], label="mission accuracy")
ax.set_xticks(_x)
ax.set_xticklabels([str(k) for k in _ks])
ax.set_xlabel("number of windows averaged, from the start of the mission")
ax.set_ylabel("mission-level score")
ax.set_ylim(0, 1)
ax.legend()
ax.set_title("Evidence accumulation on unseen days")
save_fig(fig, "7e_evidence_accumulation", "Windows are taken in chunk order from the start of each mission, so small k uses "
                                          "spin-up (start-phase) windows. Consecutive windows overlap by 50%.")

In [ ]:
# ==============================================================================
# 7f. CONTEXT ONLY (not a comparison experiment)
# ==============================================================================
_v, _lo, _hi = V28_A1_DAYOUT_F1
_in_ci = _lo <= DO_MACRO[0] <= _hi
_overlap = not (DO_MACRO[2] < _lo or DO_MACRO[1] > _hi)
print(f"This notebook, day-out pooled macro-F1: {DO_MACRO[0]:.3f} [{DO_MACRO[1]:.3f}, {DO_MACRO[2]:.3f}] "
      f"(seeds {_seeds_do}, {WINDOW_CHUNKS}-chunk windows)")
print(f"Version 28 audio arm A1, day-out pooled macro-F1: {_v:.3f} [{_lo:.3f}, {_hi:.3f}] (seeds 42/123, 2.56 s time-grid windows)")
print(f"-> {'inside' if _in_ci else 'outside'} v28's CI; CIs {'overlap' if _overlap else 'do NOT overlap'}. "
      f"This notebook's pooled macro-F1 is {'in' if _overlap else 'NOT in'} that neighbourhood. Windows, seeds and token "
      f"construction differ from v28's, so exact agreement is not expected. This is context, not a test.")

In [ ]:
# ==============================================================================
# 7g. WHAT THIS NOTEBOOK TELLS US (generated from the numbers above)
# ==============================================================================
_best_ph = PHASE_TAB["macro-F1"].idxmax()
_worst_ph = PHASE_TAB["macro-F1"].idxmin()
_worst_day = FOLD_TAB["macro-F1"].idxmin()
_best_day = FOLD_TAB["macro-F1"].idxmax()
_r, (_rlo, _rhi) = DO_RECALL[0], DO_RECALL[1]
_lines = [
    "WHAT THIS NOTEBOOK TELLS US",
    "",
    "Dataset",
    f"- {len(KEPT)} missions after excluding the 4 with corrupt recording_started_time; classes {_cls}; days {_days}.",
    f"- Each AudioBuffer record = {FRAME} frames x {N_SLOTS} interleaved slots. Slot 6 is identically zero in "
    f"{n_zero6}/{len(KEPT)} missions, and slot 1 is the loudest in {n_slot1_max}/{len(KEPT)}. So the audio is "
    f"{N_CH} channels + 1 empty slot.",
    "- There is no sample rate and no per-chunk timestamp. The ~16 kHz per channel (16 ms per buffer) rate is an "
    "UNVERIFIED hypothesis, used only for axis labels.",
    f"- Spin-up chunks are much quieter than flight (median start/steady RMS ratio {np.median(_ratio_start):.2f}).",
    f"- Day effect: day-within-class explains {VAR_SHARES['day within class'][0]:.2f} of the standardized spectral "
    f"variance (chance {VAR_SHARES['day within class'][1]:.2f}); class explains {VAR_SHARES['class'][0]:.2f} "
    f"(chance {VAR_SHARES['class'][1]:.2f}).",
    f"- Descriptively, {int(SIG.sum())} of {SIG.size} channel x bin cells pass the within-day permutation family-wise "
    f"ANOVA threshold, and {int(_strong.sum())} cells are monotone in class (|rho| >= {SPEARMAN_THRESHOLD}).",
    "",
    "Representation (fixed a priori)",
    f"- sqrt|rFFT| of per-channel bins 1-16 x 5 channels = {N_FEAT} features per chunk, no per-chunk normalization; "
    f"{WINDOW_CHUNKS}-chunk windows, stride {STRIDE_CHUNKS}, {TA} tokens; fold-train z-scoring; v28's 2-block real-Mamba "
    f"model ({N_PARAMS:,} parameters).",
    "",
    "Standalone strength",
]
if STD_RESULTS is not None:
    _lines.append(f"- STANDARD split (v24 assignment, 15 test missions): window macro-F1 {fmt_ms(STD_RESULTS['window macro_f1'])}, "
                  f"mission macro-F1 {fmt_ms(STD_RESULTS['mission macro_f1'])} over {len(SEEDS)} seeds.")
else:
    _lines.append("- STANDARD split: NOT RUN (v24's split could not be read).")
_lines += [
    f"- DAY-OUT (unseen recording days), pooled out-of-fold: window macro-F1 {DO_MACRO[0]:.3f} [95% CI {DO_MACRO[1]:.3f}, "
    f"{DO_MACRO[2]:.3f}], mission macro-F1 {DO_MMACRO[0]:.3f} [{DO_MMACRO[1]:.3f}, {DO_MMACRO[2]:.3f}]; seed-to-seed SD "
    f"of pooled macro-F1 {np.std(list(POOL_SEED_F1.values()), ddof=1) if len(POOL_SEED_F1) > 1 else float('nan'):.3f}.",
    "- Day-out per-class recall: " + ", ".join(f"c{c} {_r[c]:.2f} [{_rlo[c]:.2f}, {_rhi[c]:.2f}]" for c in range(N_CLASSES)) + ".",
    f"- Context: v28 A1 was {V28_A1_DAYOUT_F1[0]:.3f} [{V28_A1_DAYOUT_F1[1]:.3f}, {V28_A1_DAYOUT_F1[2]:.3f}]; this run is "
    f"{'in' if _overlap else 'not in'} that neighbourhood.",
    "",
    "Where it fails",
    f"- Worst unseen day: {FOLD_TAB.loc[_worst_day, 'test day']} (macro-F1 {FOLD_TAB.loc[_worst_day, 'macro-F1']:.3f}); best: "
    f"{FOLD_TAB.loc[_best_day, 'test day']} ({FOLD_TAB.loc[_best_day, 'macro-F1']:.3f}).",
    f"- By phase: best {_best_ph} (macro-F1 {PHASE_TAB.loc[_best_ph, 'macro-F1']:.3f}), worst {_worst_ph} "
    f"({PHASE_TAB.loc[_worst_ph, 'macro-F1']:.3f}).",
    "- Most confused pairs: " + "; ".join(f"{i}<->{j} ({n} windows)" for n, i, j in TOP_PAIRS) + ".",
    f"- Safety: {SAFETY['faulty windows predicted Normal (missed fault)']:.3f} of faulty windows are called Normal; "
    f"{SAFETY['Normal windows predicted faulty (false alarm)']:.3f} of Normal windows are called faulty; "
    f"{SAFETY['errors that are adjacent-class (|pred − true| = 1)']:.2f} of errors are adjacent-class.",
    f"- Evidence accumulation: mission macro-F1 {EVID.loc[1, 'mission macro-F1']:.3f} from the first window, "
    f"{EVID.loc['all', 'mission macro-F1']:.3f} with all windows.",
    "",
    "Limits",
    "- 95 missions, ~19 per class: per-class recall is only known to about ±0.1 (see the CIs above).",
    f"- {len(_seeds_do)} seeds; the 2021-12-22 fold has only 10 missions.",
    "- The per-channel sample rate is an unverified assumption, and no frequency is stated in Hz.",
    "- This is characterization, not hypothesis testing. Section 3 is descriptive and did not choose the model's inputs.",
]
SUMMARY = "\n".join(_lines)
print(SUMMARY)
(OUT / "audio_only_summary.txt").write_text(SUMMARY, encoding="utf-8")

## Final validation — every verification hook in one place
Hook 5 (the file-access summary) and hook 7 (figure caveats) are evaluated here. Hooks 1–4 and 6 were evaluated
earlier. Hooks 1–4 stopped the notebook on FAIL, before any training.

In [ ]:
# ==============================================================================
# FINAL VALIDATION — HOOK 5 (file access), HOOK 7 (figure caveats), then the table of hooks 1-7
# ==============================================================================
_parsed_now = sum(v == "parsed" for v in PARSE_STATUS.values())
_cached = sum(v == "cached" for v in PARSE_STATUS.values())
_full = FILE_LOG_PER_MISSION["AudioBuffer:full_parse"]
_sc = FILE_LOG_PER_MISSION["SensorCombined:first_last_timestamp"]
_first = FILE_LOG_PER_MISSION["AudioBuffer:first_record"]
print("File-access summary (every dataset file read goes through open_logged):")
for k in ACCESS_KINDS:
    print(f"  {k:<40} {FILE_LOG[k]:>4} opens, {len(FILE_LOG_PER_MISSION[k]):>3} distinct missions, max per mission "
          f"{max(FILE_LOG_PER_MISSION[k].values(), default=0)}")
print(f"  AudioBuffer full parses this session: {_parsed_now}; loaded from cache (parsed in an earlier session): {_cached}")
print("  The 4 excluded missions: AudioBuffer first record + SensorCombined first/last timestamp only (needed to verify "
      "the exclusion); never fully parsed.")
_h5 = (_parsed_now + _cached == EXPECTED_N_MISSIONS and FILE_LOG["AudioBuffer:full_parse"] == _parsed_now
       and max(_full.values(), default=0) <= 1 and not (set(_full) & EXPECTED_EXCLUDED)
       and len(_sc) == len(ALL_DIRS) and max(_sc.values()) == 1 and len(_first) == len(ALL_DIRS) and max(_first.values()) == 1
       and set(FILE_LOG) <= set(ACCESS_KINDS))
hook(5, _h5, f"AudioBuffer fully parsed once per kept mission ({_parsed_now} now + {_cached} cached = "
             f"{_parsed_now + _cached}/95); SensorCombined opened only for first/last timestamps "
             f"({FILE_LOG['SensorCombined:first_last_timestamp']} = 95 kept + 4 excluded); nothing else", stop=False)

_freq = [f for f in FIGURES if f["freq"]]
hook(7, all(f["freq_caveat_ok"] for f in _freq) and len(FIGURES) > 0,
     f"{len(FIGURES)} figures saved; all {len(_freq)} frequency figures carry the bin axis + unverified-Hz secondary axis "
     f"+ caveat ({[f['name'] for f in _freq]})", stop=False)

print("\n" + "=" * 110)
print("VERIFICATION HOOKS")
print("=" * 110)
for n in range(1, 8):
    res = HOOKS.get(n, [])
    status = "NOT RUN" if not res else ("PASS" if all(ok for ok, _ in res) else "FAIL")
    print(f"  HOOK {n} {status:<8} {HOOK_NAMES[n]}")
    for ok, msg in res:
        print(f"      {'PASS' if ok else 'FAIL'}: {msg}")
print("=" * 110)
print(f"Outputs: {RESULTS_CSV} | {PROBS_DIR}/ | {CKPT_DIR}/ | {FIG_DIR}/ | {OUT / 'audio_only_summary.txt'}")